<a href="https://colab.research.google.com/github/fzampirolli/pdi-vc/blob/master/notebooks_alunos/py.es/cap07/cap07.EPs_aluno.ipynb"><img src="imagens/colab-badge.png" style="height:20px;vertical-align:middle"></a>
<a href="https://github.com/fzampirolli/pdi-vc"><img src="imagens/github-badge.png" style="height:20px;vertical-align:middle"></a>

## 💻 **Parte Práctica con Ejercicios de Programación**

La presente lista de ejercicios de programación (EP) consolida las formulaciones teóricas presentadas a lo largo del Capítulo 7 — Clasificación de Imágenes y Reconocimiento de Patrones — mediante una ruta práctica aplicada. A diferencia de la manipulación directa de píxeles de los capítulos anteriores, los EPs de este capítulo trabajan con las **magnitudes intermedias** de un *pipeline* real de reconocimiento de patrones — vectores de características, distancias, etiquetas previstas y reales, códigos binarios locales e histogramas de orientación — permitiendo validar manualmente cada etapa del razonamiento sin depender de bibliotecas externas de aprendizaje automático.

El encadenamiento de los ejercicios reproduce el flujo conceptual del capítulo: se comienza con la implementación manual de la regla de decisión del clasificador **k-NN** sobre un pequeño espacio de características; a continuación, se revisita, desde la óptica de la normalización de características, el clasificador implementado en el primer ejercicio de la lista; se avanza hacia el cálculo de las métricas de **evaluación** (matriz de confusión, precisión y exhaustividad) a partir de etiquetas previstas y reales; se prosigue con la codificación manual del descriptor de textura **LBP** a partir de una vecindad $3\times3$; se profundiza en el cálculo del histograma de orientaciones del descriptor **HOG** para una única celda; se avanza, a continuación, hacia la integración de **extracción de descriptores**, **clasificación k-NN** y **evaluación multiclase** en un *pipeline* completo de reconocimiento de texturas; y se concluye con la aplicación de ese mismo *pipeline* sobre una **imagen real** (formato PGM), en el que el descriptor LBP se calcula directamente sobre los píxeles de un mosaico de texturas.

### 🎯 Objetivo de este Cuaderno

El cuaderno permite desarrollar, validar, organizar y probar soluciones de **Ejercicios de Programación (EPs)** en entornos interactivos, como Colab, con los mismos casos de prueba de Moodle, copiándolos allí solo al momento de registrar la nota oficial.

### *Download*

Descargue `morph.py` y `testsuite.py` ejecutando la celda siguiente:

In [ ]:
import os, urllib.request

url = "https://raw.githubusercontent.com/fzampirolli/pdi-vc/master/morph/config.py"
if not os.path.exists("config.py"):
    urllib.request.urlretrieve(url, "config.py")

import config
config.setup(testsuite=True)
from morph import mm
from testsuite import TestSuite

#### Ejecutando las pruebas
Para evaluar las pruebas, ejecute `TestSuite("EP07_01.extensión").run()` en una nueva celda, reemplazando la extensión por la del lenguaje utilizado (`.py`, `.java`, `.c`, `.cpp`, `.js` o `.r`). El sistema descarga los casos de prueba de GitHub, ejecuta el programa y calcula la nota automáticamente.

Para probar código Python directamente, sin guardar archivo, use `run_code(codigo)` pasando el código como *string* en una variable `codigo`:

```python
codigo = """
# ... su código aquí ...
"""
TestSuite("EP07_01").run_code(codigo)
```

### 🛠️ Resumen de los Métodos de `morph.py` (Cap. 7)

La biblioteca `morph.py` ofrece dos versiones para la mayoría de los algoritmos: una **didáctica** (métodos terminados en `0`), implementada paso a paso en NumPy, y otra **clásica**, basada en las bibliotecas `scikit-learn` y `scikit-image`. Las implementaciones didácticas se utilizan en los **Ejercicios de Programación (EPs)**, ya que no dependen de bibliotecas externas y se ejecutan dentro del límite de memoria del entorno **VPL** de Moodle. En cambio, las versiones clásicas son más eficientes y se recomiendan para experimentos en entornos como Colab y Jupyter Notebook, pero normalmente **no pueden utilizarse en los EPs** de Moodle, ya que la biblioteca `scikit-learn` excede la memoria disponible en el VPL.

1. **Lectura de datos (`readClasses`, `readDataset`, `readTrain`, `readTest`)**  
   Estandarizan la entrada de los conjuntos de entrenamiento y prueba, devolviendo las matrices de características ($X$) y los vectores de etiquetas ($y$).

2. **Clasificación (`knn0` / `knn`)**  
   Implementan el algoritmo de los **k-vecinos más cercanos (k-NN)** para clasificación binaria y multiclase, utilizando distancia Euclidiana o Manhattan.

3. **Normalización (`zscore0` / `zscore`)**  
   Aplican la normalización *z-score* a los atributos, reduciendo diferencias de escala antes de la clasificación.

4. **Evaluación (`confusion0` / `confusion`)**  
   Calculan la matriz de confusión y métricas como exactitud, precisión y recuperación, tanto para problemas binarios como multiclase.

5. **Descriptor de textura (`lbp0` / `lbp`)**  
   Calculan el ***Local Binary Pattern* (LBP)**, permitiendo obtener el mapa LBP, el código de un píxel o el histograma de una región de la imagen.

6. **Descriptor de forma (`hog0` / `hog`)**  
   Calculan el ***Histogram of Oriented Gradients* (HOG)**, produciendo histogramas de las orientaciones de los gradientes para representar información de forma y contorno.

### EP07_01 🟢 Clasificador k-NN Paso a Paso

El `KNeighborsClassifier` de `scikit-learn`, utilizado a lo largo del capítulo, oculta detrás de una única llamada (`.fit` / `.predict`) una regla de decisión bastante simple: para cada nueva observación, calcular la distancia a todos los ejemplos de entrenamiento, seleccionar los $k$ más cercanos y votar por la clase mayoritaria entre ellos.

Antes de confiar en la biblioteca, se te ha encargado implementar esta regla desde cero, para un espacio de características bidimensional, exactamente como el simulador interactivo de frontera de decisión del capítulo lo hace internamente en cada clic del usuario.

#### 📋 Directrices de Implementación

1. **Cantidad y parámetro:** Leer el entero $N$ (número de ejemplos de entrenamiento) y el entero impar $k$ (número de vecinos).
2. **Ejemplos de entrenamiento:** Para cada uno de los $N$ ejemplos, leer tres valores: las coordenadas $x$ e $y$ (reales) y la etiqueta $r$ (entera, $0$ o $1$).
3. **Consultas:** Leer el entero $Q$ (número de puntos de consulta) y, a continuación, las coordenadas $x_q$, $y_q$ (reales) de cada consulta.
4. **Distancia:** Para cada consulta, calcular la distancia euclidiana hasta **todos** los ejemplos de entrenamiento:
$$
d(x_q, x_i) = \sqrt{(x_q - x_i)^2 + (y_q - y_i)^2}.
$$
5. **Selección de vecinos:** Ordenar los ejemplos por distancia creciente y seleccionar los $k$ primeros. En caso de **empate de distancia** en la frontera del k-ésimo vecino, desempatar por el ejemplo leído **primero** en la entrada (orden de lectura estable).
6. **Votación mayoritaria:** Contar los votos de cada clase entre los $k$ vecinos seleccionados. Si hay **empate en la votación** (solo posible cuando $k$ es par, lo que no debería ocurrir según la directriz del punto 1, pero tratar defensivamente), asignar la clase del vecino más cercano entre las clases empatadas.
7. **Salida:** Para cada consulta, en el orden de entrada, imprimir la clase predicha. Al final, imprimir el total de consultas clasificadas como clase `1`.

#### 📌 Restricciones Computacionales

* **Métrica fija:** utilizar exclusivamente la distancia euclidiana (no la *distancia al cuadrado*) para la ordenación, aunque el resultado de la comparación sea el mismo.
* **k siempre impar:** la entrada garantiza $k$ impar y $k \le N$; aun así, implementar el desempate del punto 6 por robustez.
* **Estabilidad:** al ordenar por distancia, preservar el orden relativo de ejemplos con la misma distancia (ordenación estable).

#### 🧠 Fundamentación Teórica

| Elemento | Papel en el k-NN |
|---|---|
| Espacio de características | Conjunto de todos los vectores $(x, y)$ posibles |
| Distancia euclidiana | Medida de similitud entre observaciones |
| $k$ pequeño | Frontera irregular, alta varianza |
| $k$ grande | Frontera suave, alto sesgo |
| Votación mayoritaria | Regla de decisión $\hat y = \operatorname{moda}\{y_i : x_i \in N_k(x)\}$ |

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Enteros $N$ y $k$, separados por espacio.
* Siguientes $N$ líneas: tres valores por línea — $x$, $y$ (reales) y $r$ (entero $\in \{0,1\}$), separados por espacio.
* Siguiente línea: entero $Q$.
* Siguientes $Q$ líneas: dos valores por línea — $x_q$, $y_q$ (reales), separados por espacio.

**Salida:**

* $Q$ líneas, cada una con la clase predicha (`0` o `1`) para la respectiva consulta, en el orden de entrada.
* Última línea: `Total clase 1: X`.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 4 3<br>0 0 0<br>1 0 0<br>5 5 1<br>6 5 1<br>1<br>1 1 | 0<br>Total clase 1: 0 | Consulta cercana al agrupamiento de clase 0. |
| 4 1<br>0 0 0<br>1 0 0<br>5 5 1<br>6 5 1<br>2<br>0.9 0.1<br>5.5 5.1 | 0<br>1<br>Total clase 1: 1 | Con $k=1$, cada consulta hereda la clase del vecino más cercano. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0701" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0701 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0701 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0701 button:hover { background: #e8dfcf; }
  #sim-ep0701 input[type=range] { width: 100%; accent-color: #26241d; cursor: pointer; height: 4px; }
  .sim-ep0701_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0701_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP07_01: Clasificador k-NN Paso a Paso</span>
  <span class="sim-ep0701_pill">Votación Mayoritaria</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Controles -->
  <div class="sim-ep0701_panel" style="margin-bottom:14px;">
    <div style="display:flex; justify-content:space-between; align-items:center; margin-bottom:6px;">
      <label style="font-size:11px; font-weight:700; color:#5e5a4a;">
        Número de Vecinos (k): <span id="sim-ep0701_vl" style="font-family:monospace; color:#26241d;">3</span>
      </label>
    </div>
    
    <input id="sim-ep0701_sl" type="range" min="1" max="7" step="2" value="3">
    
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; margin-top:8px; text-align:center;">
      Ajusta k y observa qué ejemplos de entrenamiento (ordenados por distancia) participan en la votación para la consulta fija (&starf; en x = 3, y = 3).
    </div>
  </div>

  <!-- Cards de Amostras -->
  <div id="sim-ep0701_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0701_debug" class="sim-ep0701_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep01(root){
    if (!root || root.dataset.sim07Ep01Init) return;
    root.dataset.sim07Ep01Init = "1";

    var query = {x: 3, y: 3};
    var pontos = [
      {nome: "A", x: 0, y: 0, r: 0},
      {nome: "B", x: 1, y: 0, r: 0},
      {nome: "C", x: 5, y: 5, r: 1},
      {nome: "D", x: 6, y: 5, r: 1},
      {nome: "E", x: 2, y: 2, r: 0},
      {nome: "F", x: 4, y: 4, r: 1},
      {nome: "G", x: 0, y: 2, r: 0},
      {nome: "H", x: 6, y: 3, r: 1}
    ];

    pontos.forEach(function(p, i){
      p.d = Math.sqrt(Math.pow(p.x - query.x, 2) + Math.pow(p.y - query.y, 2));
      p.idx = i;
    });

    pontos.sort(function(a, b){
      return (a.d - b.d) || (a.idx - b.idx);
    });

    var slEl  = root.querySelector('#sim-ep0701_sl');
    var vlEl  = root.querySelector('#sim-ep0701_vl');
    var cards = root.querySelector('#sim-ep0701_cards');
    var dbg   = root.querySelector('#sim-ep0701_debug');

    function render(){
      var k = parseInt(slEl.value, 10);
      vlEl.textContent = k;
      cards.innerHTML = '';
      var votos = [0, 0];

      pontos.forEach(function(p, i){
        var dentro = i < k;
        if (dentro) votos[p.r]++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? (p.r === 0 
                ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
                : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;') 
            : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;');

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + p.nome + ' (r = ' + p.r + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">d = ' + p.d.toFixed(2) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'VOTA' : '&ndash;') + '</div>';

        cards.appendChild(div);
      });

      var previsto = votos[1] > votos[0] ? 1 : (votos[0] > votos[1] ? 0 : pontos[0].r);

      if (previsto === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'k = ' + k + '  |  Votos Classe 0: ' + votos[0] + ', Classe 1: ' + votos[1] + '  |  Classe prevista: ' + previsto;
    }

    slEl.addEventListener('input', render);
    render();
  }

  function tryInitSim07Ep01(){
    var root = document.getElementById('sim-ep0701');
    if (root) initSim07Ep01(root); else setTimeout(tryInitSim07Ep01, 200);
  }
  tryInitSim07Ep01();
})();
</script>
""")

**Figura 7.1:** Simulador EP07_01: Clasificador k-NN Paso a Paso


In [ ]:
%%writefile EP07_01.py
# Código Python

In [ ]:
TestSuite("EP07_01.py").run()

### EP07_02 🟡 Normalización *Z-score* y Robustez del k-NN ante Escalas Distintas

Este ejercicio revisita el clasificador implementado en el **EP07_01**, esta vez bajo la óptica discutida en la sección *El Impacto de la Escala y la Normalización de Características* del capítulo: el k-NN decide basándose en la distancia entre vectores, de modo que una característica medida en una escala mucho mayor que las demás tiende a **dominar** el cálculo de la distancia, incluso cuando no es la más relevante para separar las clases.

Un sistema de inspección registra, para cada pieza, su **área** (en píxeles, pudiendo llegar a cientos o miles) y su **circularidad** (siempre entre $0$ y $1$). Usted ha sido encargado de clasificar nuevas piezas mediante k-NN de dos formas — con y sin la estandarización *Z-score* presentada en el capítulo — y de reportar en qué casos ambas aproximaciones **divergen**.

#### 📋 Directrices de Implementación

1. **Cantidad y parámetro:** Leer el entero $N$ (número de ejemplos de entrenamiento) y el entero impar $k$.
2. **Ejemplos de entrenamiento:** Para cada uno de los $N$ ejemplos, leer tres valores: el área $x_1$ (real), la circularidad $x_2$ (real) y la etiqueta $r$ (entero, $0$ o $1$).
3. **Consultas:** Leer el entero $Q$ y, a continuación, las coordenadas $x_1, x_2$ de cada consulta.
4. **Clasificación sin normalización:** Para cada consulta, clasifíquela mediante k-NN directamente sobre $(x_1, x_2)$, con distancia euclidiana y las mismas reglas de desempate del EP07_01 (orden de lectura para distancias empatadas; vecino más cercano entre clases empatadas en la votación).
5. **Parámetros de normalización:** Calcular la media $\mu_j$ y la desviación estándar **poblacional** $\sigma_j$ (división por $N$, no por $N-1$ — la misma convención adoptada por la clase `StandardScaler`) de cada característica $j \in \{1,2\}$, **exclusivamente sobre el conjunto de entrenamiento**.
6. **Estandarización:** Transformar cada característica de entrenamiento y de consulta mediante
$$
z_j = \frac{x_j - \mu_j}{\sigma_j}.
$$
Si $\sigma_j = 0$ (característica constante en el entrenamiento), definir $z_j = 0$ para todas las muestras de esa característica, evitando la división por cero.
7. **Clasificación con normalización:** Repetir la clasificación k-NN del punto 4, ahora sobre los vectores estandarizados $(z_1, z_2)$, con las mismas reglas de desempate.
8. **Salida:** Para cada consulta, en el orden de entrada, imprimir las dos clases previstas. Al final, imprimir el número de consultas en las que las dos clasificaciones **divergen**.

#### 📌 Restricciones Computacionales

* **Ajuste solo en el entrenamiento:** $\mu_j$ y $\sigma_j$ se calculan únicamente a partir del conjunto de entrenamiento y se reaplican a las consultas — nunca se recalculan a partir de ellas. Esta práctica evita la **fuga de datos** (*data leakage*), mencionada en la sección de normalización del capítulo.
* **Desviación estándar poblacional:** utilizar $\sigma_j = \sqrt{\frac{1}{N}\sum_i (x_{i,j}-\mu_j)^2}$, y no la versión muestral (división por $N-1$).
* **Característica constante:** tratar $\sigma_j = 0$ como caso especial (punto 6); no debe ocurrir un error de división por cero.
* **Reglas de desempate:** reutilizar exactamente las convenciones del EP07_01, tanto en la selección de los $k$ vecinos como en la votación mayoritaria.

#### 🧠 Fundamentación Teórica

| Elemento | Papel |
|---|---|
| Estandarización *Z-score* | Reescala cada característica para media $0$ y desviación estándar $1$, haciendo comparables escalas heterogéneas |
| Ajuste (*fit*) solo en el entrenamiento | Garantiza que la evaluación sobre las consultas refleje únicamente lo que el modelo aprendió en el entrenamiento |
| Distancia euclidiana sin normalización | Dominada por la característica de mayor amplitud — aquí, el área |
| Predicción divergente | Evidencia que la escala de las características, y no solo el algoritmo o los datos, puede determinar la frontera de decisión del k-NN |

Este ejercicio refuerza, de forma controlada, la razón por la cual el `StandardScaler` se aplica antes del k-NN a lo largo del capítulo: sin esta etapa, las características de circularidad — incluso siendo altamente discriminativas — pueden ser prácticamente ignoradas por el clasificador frente a una característica de área con amplitud cientos de veces mayor.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Enteros $N$ y $k$, separados por espacio.
* Siguientes $N$ líneas: tres valores por línea — $x_1$, $x_2$ (reales) y $r$ (entero $\in \{0,1\}$), separados por espacio.
* Siguiente línea: entero $Q$.
* Siguientes $Q$ líneas: dos valores por línea — $x_1$, $x_2$ (reales) de la consulta, separados por espacio.

**Salida:**

* $Q$ líneas, en el formato `SemNorm=<0|1> ComNorm=<0|1>`, en el orden de entrada de las consultas.
* Última línea: `Divergiu: <int>`.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 4 3<br>10 0.9 0<br>12 0.85 0<br>900 0.2 1<br>950 0.25 1<br>1<br>500 0.88 | SemNorm=1 ComNorm=0<br>Divergiu: 1 | Sin normalización, el área (escala de cientos) domina la distancia y la consulta se clasifica como clase `1`. Tras la estandarización, la circularidad — mucho más cercana a las muestras de clase `0` — pasa a pesar de forma comparable, y la predicción cambia a `0`. |
| 2 1<br>0 0.5 0<br>100 0.5 1<br>1<br>60 0.5 | SemNorm=1 ComNorm=1<br>Divergiu: 0 | La circularidad es constante en el entrenamiento ($\sigma_2=0$); por la regla del punto 6, $z_2=0$ para todas las muestras, y la clasificación depende solo del área en ambos casos. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0702" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0702 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0702 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0702 button:hover { background: #e8dfcf; }
  #sim-ep0702 button.sim-ep0702_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0702_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0702_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP07_02: Normalización Z-score y Distancia k-NN</span>
  <span class="sim-ep0702_pill">Estandarización de Características</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Descrição e Seleção de Modo -->
  <div class="sim-ep0702_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:8px;">
      Cada ejemplo posee dos características: área (px) y circularidad [0, 1]. Alterne la normalización y observe el cambio en la clase prevista.
    </div>
    
    <div id="sim-ep0702_query" style="font-size:11px; color:#26241d; text-align:center; font-family:monospace; font-weight:700; margin-bottom:10px;"></div>

    <div style="display:flex; justify-content:center; gap:8px; flex-wrap:wrap;">
      <button id="sim-ep0702_btn_raw" class="sim-ep0702_active">Sin Normalización</button>
      <button id="sim-ep0702_btn_norm">Con Normalización (Z-score)</button>
    </div>
  </div>

  <!-- Cards de Amostras -->
  <div id="sim-ep0702_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0702_debug" class="sim05_ep01_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center; background:#fafaf7; border:1px solid #e9e3d3; border-radius:12px; padding:12px;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep02(root){
    if (!root || root.dataset.sim07Ep02Init) return;
    root.dataset.sim07Ep02Init = "1";

    var pontos = [
      {nome: "P1", x1: 10,  x2: 0.90, r: 0},
      {nome: "P2", x1: 12,  x2: 0.85, r: 0},
      {nome: "P3", x1: 900, x2: 0.20, r: 1},
      {nome: "P4", x1: 950, x2: 0.25, r: 1}
    ];

    pontos.forEach(function(p, i){ p.idx = i; });
    var query = {x1: 500, x2: 0.88};
    var k = 3;

    function stats(vals){
      var m = vals.reduce(function(a, b){ return a + b; }, 0) / vals.length;
      var v = vals.reduce(function(a, s){ return a + (s - m) * (s - m); }, 0) / vals.length;
      return {mean: m, std: Math.sqrt(v)};
    }

    var s1 = stats(pontos.map(function(p){ return p.x1; }));
    var s2 = stats(pontos.map(function(p){ return p.x2; }));

    function z(x, s){ return s.std === 0 ? 0 : (x - s.mean) / s.std; }

    var cards   = root.querySelector('#sim-ep0702_cards');
    var dbg     = root.querySelector('#sim-ep0702_debug');
    var qEl     = root.querySelector('#sim-ep0702_query');
    var btnRaw  = root.querySelector('#sim-ep0702_btn_raw');
    var btnNorm = root.querySelector('#sim-ep0702_btn_norm');
    var modoNorm = false;

    function render(){
      btnRaw.classList.toggle('sim-ep0702_active', !modoNorm);
      btnNorm.classList.toggle('sim-ep0702_active', modoNorm);

      qEl.textContent = '★ Consulta: Área = ' + query.x1 + ', Circularidade = ' + query.x2 +
        (modoNorm ? ' → z_área = ' + z(query.x1, s1).toFixed(3) + ', z_circ = ' + z(query.x2, s2).toFixed(3) : '');

      var qx1 = modoNorm ? z(query.x1, s1) : query.x1;
      var qx2 = modoNorm ? z(query.x2, s2) : query.x2;

      var lista = pontos.map(function(p){
        var px1 = modoNorm ? z(p.x1, s1) : p.x1;
        var px2 = modoNorm ? z(p.x2, s2) : p.x2;
        var d = Math.sqrt((px1 - qx1) * (px1 - qx1) + (px2 - qx2) * (px2 - qx2));
        return {nome: p.nome, r: p.r, d: d, idx: p.idx, area: p.x1, circ: p.x2, va: px1, vc: px2};
      });

      lista.sort(function(a, b){ return (a.d - b.d) || (a.idx - b.idx); });

      cards.innerHTML = '';
      var votos = [0, 0];

      lista.forEach(function(p, i){
        var dentro = i < k;
        if (dentro) votos[p.r]++;

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' +
          (dentro 
            ? (p.r === 0 
                ? 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;' 
                : 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;') 
            : 'background:#fafaf7; border:1px solid #e4dcc8; color:#8a8371;');

        var valorUsado = modoNorm
          ? ('z = (' + p.va.toFixed(2) + ', ' + p.vc.toFixed(2) + ')')
          : ('área = ' + p.area + ', circ = ' + p.circ);

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">' + p.nome + ' (r = ' + p.r + ')</div>' +
          '<div style="font-family:monospace; margin-bottom:4px; font-size:10px;">' + valorUsado + '</div>' +
          '<div style="font-family:monospace; margin-bottom:4px;">d = ' + p.d.toFixed(3) + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + (dentro ? 'VOTA' : '&ndash;') + '</div>';

        cards.appendChild(div);
      });

      var previsto = votos[1] > votos[0] ? 1 : (votos[0] > votos[1] ? 0 : lista[0].r);

      if (previsto === 0) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = (modoNorm ? 'COM Normalização' : 'SEM Normalização') +
        '  |  k = ' + k + '  |  Votos Classe 0: ' + votos[0] + ', Classe 1: ' + votos[1] +
        '  |  Classe prevista: ' + previsto;
    }

    btnRaw.addEventListener('click', function(){ modoNorm = false; render(); });
    btnNorm.addEventListener('click', function(){ modoNorm = true; render(); });
    render();
  }

  function tryInitSim07Ep02(){
    var root = document.getElementById('sim-ep0702');
    if (root) initSim07Ep02(root); else setTimeout(tryInitSim07Ep02, 200);
  }
  tryInitSim07Ep02();
})();
</script>
""")

**Figura 7.2:** Simulador EP07_02: Efecto de la Normalización *Z-score* en la Distancia k-NN


In [ ]:
%%writefile EP07_02.py
# Código Python

In [ ]:
TestSuite("EP07_02.py").run()

### EP07_03 🟡 Evaluación por Matriz de Confusión

Un clasificador binario de calidad de soldadura fue entrenado y probado en una línea de producción. Para cada pieza inspeccionada, el sistema registró la etiqueta **real** (obtenida por un especialista) y la etiqueta **predicha** por el clasificador, donde `1` representa "defectuosa" y `0` representa "conforme".

La gerencia de calidad quiere saber no solo la exactitud del sistema, sino también su **precisión** (cuando el sistema señala un defecto, ¿con qué frecuencia está en lo correcto?) y su **revocación** (de todas las piezas realmente defectuosas, ¿cuántas logró identificar el sistema?) — la distinción discutida en la sección de evaluación de clasificadores del capítulo.

#### 📋 Directrices de Implementación

1. **Cantidad:** Leer el entero $N$ (número de piezas inspeccionadas).
2. **Datos de cada pieza:** Para cada una de las $N$ piezas, leer dos enteros — la etiqueta real $y$ y la etiqueta predicha $\hat y$ (ambas $\in \{0, 1\}$).
3. **Matriz de confusión:** Considerando la clase `1` (defectuosa) como **positiva**, contar:
   - $VP$ (Verdadero Positivo): $y=1$ y $\hat y=1$;
   - $FP$ (Falso Positivo): $y=0$ y $\hat y=1$;
   - $FN$ (Falso Negativo): $y=1$ y $\hat y=0$;
   - $VN$ (Verdadero Negativo): $y=0$ y $\hat y=0$.
4. **Métricas:** Calcular
$$
\text{Exactitud} = \frac{VP+VN}{N}, \quad
\text{Precisión} = \frac{VP}{VP+FP}, \quad
\text{Revocación} = \frac{VP}{VP+FN}.
$$
5. **Casos degenerados:** Si $VP+FP=0$ (ninguna predicción positiva), imprimir `Precisao: indefinida`. Si $VP+FN=0$ (ningún caso positivo real), imprimir `Revocacao: indefinida`.
6. **Redondeo:** Todas las métricas numéricas deben redondearse a 4 decimales (*redondeo half away from zero*) solo en la visualización.

#### 📌 Restricciones Computacionales

* **Convención de clase positiva fija:** la clase `1` es siempre la clase positiva en este ejercicio, independientemente de su frecuencia relativa.
* **Protección contra división por cero:** implemente los casos degenerados del punto 5 antes de realizar la división.
* **Orden de salida:** siga exactamente el orden especificado en la sección de salida, incluso en los casos degenerados.

#### 🧠 Fundamentación Teórica

| Métrica | Pregunta que responde | ¿Sensible a desbalanceo? |
|---|---|---|
| Exactitud | ¿Qué fracción de las piezas fue clasificada correctamente? | Sí — puede enmascarar errores en la clase minoritaria |
| Precisión | De las piezas señaladas como defectuosas, ¿cuántas realmente lo son? | Penaliza falsos positivos |
| Revocación | De las piezas realmente defectuosas, ¿cuántas fueron detectadas? | Penaliza falsos negativos |

En un contexto industrial, una **revocación** baja es frecuentemente más grave que una **precisión** baja: dejar pasar una pieza defectuosa (falso negativo) tiende a ser más costoso que inspeccionar manualmente una pieza buena señalada por error (falso positivo).

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $N$.
* Siguientes $N$ líneas: dos enteros por línea — $y$ y $\hat y$, separados por espacio.

**Salida (en este orden exacto):**

```
VP=<int> FP=<int> FN=<int> VN=<int>
Acuracia: <valor o métrica indefinida>
Precisao: <valor o indefinida>
Revocacao: <valor o indefinida>
```

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 4<br>1 1<br>0 1<br>1 0<br>0 0 | VP=1 FP=1 FN=1 VN=1<br>Acuracia: 0.5000<br>Precisao: 0.5000<br>Revocacao: 0.5000 | Un error de cada tipo. |
| 3<br>0 0<br>0 0<br>0 0 | VP=0 FP=0 FN=0 VN=3<br>Acuracia: 1.0000<br>Precisao: indefinida<br>Revocacao: indefinida | Ningún caso positivo real ni predicho. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0703" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0703 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0703 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0703 button:hover { background: #e8dfcf; }
  #sim-ep0703 button.sim-ep0703_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0703_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0703_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP07_03: Precisión x Revocación</span>
  <span class="sim-ep0703_pill">Línea de Producción</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Seleção de Cenário -->
  <div class="sim-ep0703_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Elija un escenario de inspección y observe cómo reaccionan de manera diferente la Exactitud, la Precisión y la Revocación.
    </div>

    <div style="display:flex; gap:6px; flex-wrap:wrap; justify-content:center;">
      <button id="sim-ep0703_b1" class="sim-ep0703_active">Escenario A: Errores Equilibrados</button>
      <button id="sim-ep0703_b2">Escenario B: Falsos Negativos</button>
      <button id="sim-ep0703_b3">Escenario C: Sin Defecto Real</button>
      <button id="sim-ep0703_b4">Escenario D: Falsos Positivos</button>
    </div>
  </div>

  <!-- Cards das Peças do Cenário -->
  <div id="sim-ep0703_cards" style="display:grid; grid-template-columns: repeat(auto-fit, minmax(110px, 1fr)); gap:10px; margin-bottom:14px;"></div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0703_debug" class="sim-ep0703_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep03(root){
    if (!root || root.dataset.sim07Ep03Init) return;
    root.dataset.sim07Ep03Init = "1";

    var cenarios = {
      A: [{y:1, p:1}, {y:0, p:1}, {y:1, p:0}, {y:0, p:0}],
      B: [{y:1, p:0}, {y:1, p:0}, {y:1, p:1}, {y:0, p:0}],
      C: [{y:0, p:0}, {y:0, p:0}, {y:0, p:0}],
      D: [{y:0, p:1}, {y:0, p:1}, {y:1, p:1}, {y:0, p:0}]
    };

    var cards = root.querySelector('#sim-ep0703_cards');
    var dbg   = root.querySelector('#sim-ep0703_debug');

    var botoes = {
      A: root.querySelector('#sim-ep0703_b1'),
      B: root.querySelector('#sim-ep0703_b2'),
      C: root.querySelector('#sim-ep0703_b3'),
      D: root.querySelector('#sim-ep0703_b4')
    };

    function render(key){
      Object.keys(botoes).forEach(function(k){
        botoes[k].classList.toggle('sim-ep0703_active', k === key);
      });

      var dados = cenarios[key];
      var VP = 0, FP = 0, FN = 0, VN = 0;
      cards.innerHTML = '';

      dados.forEach(function(d, i){
        if (d.y === 1 && d.p === 1) VP++;
        else if (d.y === 0 && d.p === 1) FP++;
        else if (d.y === 1 && d.p === 0) FN++;
        else VN++;

        var statusCor = '';
        var statusTxt = '';

        if (d.y === d.p) {
          statusCor = 'background:#eafaf1; border:1px solid #a3e4d7; color:#04342C;';
          statusTxt = d.y === 1 ? 'VP (Acerto)' : 'VN (Acerto)';
        } else {
          statusCor = 'background:#fdecea; border:1px solid #f5b7b1; color:#c0392b;';
          statusTxt = d.p === 1 ? 'FP (Alarme Falso)' : 'FN (Escapou)';
        }

        var div = document.createElement('div');
        div.style.cssText = 'text-align:center; border-radius:10px; padding:10px 6px; font-size:11px; transition:all 0.15s ease;' + statusCor;

        div.innerHTML = '<div style="font-weight:700; margin-bottom:4px;">Peça ' + (i + 1) + '</div>' +
          '<div style="font-family:monospace; font-size:10px; margin-bottom:4px;">Real = ' + d.y + ' | Prev = ' + d.p + '</div>' +
          '<div style="font-weight:700; font-size:10px;">' + statusTxt + '</div>';

        cards.appendChild(div);
      });

      var N = dados.length;
      var acc = ((VP + VN) / N).toFixed(4);
      var prec = (VP + FP) > 0 ? (VP / (VP + FP)).toFixed(4) : 'Indefinida';
      var rev = (VP + FN) > 0 ? (VP / (VP + FN)).toFixed(4) : 'Indefinida';

      if (prec === 'Indefinida' || parseFloat(prec) < 0.5) {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      } else {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      }

      dbg.textContent = 'VP = ' + VP + ' | FP = ' + FP + ' | FN = ' + FN + ' | VN = ' + VN +
        '  |  Acurácia = ' + acc + '  |  Precisão = ' + prec + '  |  Revocação = ' + rev;
    }

    botoes.A.addEventListener('click', function(){ render('A'); });
    botoes.B.addEventListener('click', function(){ render('B'); });
    botoes.C.addEventListener('click', function(){ render('C'); });
    botoes.D.addEventListener('click', function(){ render('D'); });

    render('A');
  }

  function tryInitSim07Ep03(){
    var root = document.getElementById('sim-ep0703');
    if (root) initSim07Ep03(root); else setTimeout(tryInitSim07Ep03, 200);
  }
  tryInitSim07Ep03();
})();
</script>
""")

**Figura 7.3:** Simulador EP07_03: Precisión x Revocación


In [ ]:
%%writefile EP07_03.py
# Código Python

In [ ]:
TestSuite("EP07_03.py").run()

### EP07_04 🟠 Codificación Manual del Descriptor LBP

La función `local_binary_pattern` de `scikit-image`, utilizada en el proyecto de clasificación de texturas, calcula automáticamente el código LBP de cada píxel de una imagen. Antes de utilizarla como una caja negra, se le ha encargado implementar manualmente el cálculo del código LBP clásico ($P=8$, $R=1$) para el píxel central de una vecindad $3\times3$, exactamente como se define en la ecuación del capítulo.

Además del código, el sistema de inspección de texturas también necesita saber si ese patrón es **uniforme** — un patrón es uniforme cuando el número de transiciones ($0\to1$ o $1\to0$) al recorrer los 8 bits **circularmente** (volviendo del último bit al primero) es **como máximo 2**, propiedad explorada por la variante *uniforme* del LBP mencionada en el capítulo.

#### 📋 Directrices de Implementación

1. **Cantidad:** Leer el entero $T$ (número de vecindades a procesar).
2. **Datos de cada vecindad:** Para cada una de las $T$ vecindades, leer una matriz $3\times3$ de enteros (intensidades), proporcionada en 3 líneas de 3 valores cada una. El píxel central es la posición `[1][1]`.
3. **Orden de los vecinos:** Recorra los 8 vecinos en sentido **horario**, comenzando en la esquina superior izquierda, en el siguiente orden de posiciones `[fila][columna]`: `[0][0]`, `[0][1]`, `[0][2]`, `[1][2]`, `[2][2]`, `[2][1]`, `[2][0]`, `[1][0]`. Este es el índice $p = 0, 1, \ldots, 7$ de la ecuación del LBP.
4. **Función umbral:** Para cada vecino $p$ con intensidad $g_p$ y centro $g_c$, calcule $s(g_p - g_c)$, que vale `1` si $g_p \geq g_c$ y `0` en caso contrario.
5. **Código LBP:** Calcule
$$
\mathrm{LBP} = \sum_{p=0}^{7} s(g_p - g_c)\, 2^p.
$$
6. **Transiciones:** Considerando la secuencia circular de bits $s_0, s_1, \ldots, s_7$ (en el orden del punto 3), cuente cuántos pares consecutivos **adyacentes en la secuencia circular** (incluyendo el par $s_7, s_0$) difieren entre sí.
7. **Clasificación:** Si el número de transiciones es $\le 2$, clasifique como `UNIFORME`; en caso contrario, `NAO_UNIFORME`.
8. **Salida:** Para cada vecindad, en el orden de entrada, imprimir el código LBP (entero decimal, $0$–$255$), el número de transiciones y la clasificación.

#### 📌 Restricciones Computacionales

* **Orden fija de los vecinos:** el orden del punto 3 es obligatorio — invertirlo produce un código numéricamente diferente, incluso representando el mismo patrón visual.
* **Comparación no estricta:** $s(z) = 1$ cuando $z \ge 0$ (el propio capítulo define la igualdad como incluida en el caso `1`).
* **Conteo circular:** no olvide el par que cierra el ciclo ($s_7$ con $s_0$); ignorar ese par es un error común que clasifica incorrectamente patrones uniformes.

#### 🧠 Fundamentación Teórica

| Patrón (bits $s_0\ldots s_7$) | Transiciones | Interpretación |
|---|---|---|
| `00000000` o `11111111` | 0 | Región homogénea (mancha clara u oscura) |
| `00001111` | 2 | Borde simple entre dos regiones |
| `01010101` | 8 | Textura de contraste alternado — no uniforme |

Los patrones uniformes se concentran en regiones de textura suave o bordes simples; los patrones no uniformes tienden a corresponder a ruido de alta frecuencia. Por eso, el histograma LBP *uniforme*, usado en el proyecto de clasificación de texturas, agrupa todos los patrones no uniformes en un único compartimento, reduciendo la dimensionalidad del descriptor.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Entero $T$.
* Para cada vecindad: 3 líneas con 3 enteros cada una (matriz $3\times3$).

**Salida:**

* $T$ líneas, en el formato `LBP=<int> transicoes=<int> <UNIFORME|NAO_UNIFORME>`.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 1<br>10 10 10<br>10 50 10<br>10 10 10 | LBP=0 transicoes=0 UNIFORME | El centro es el más claro; todos los vecinos generan bit 0. |
| 1<br>90 90 90<br>10 50 10<br>90 90 90 | LBP=119 transicoes=4 NAO_UNIFORME | Vecinos claros y oscuros alternados en la vecindad. |

In [ ]:
from IPython.display import HTML
HTML("""
<div id="sim-ep0704" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">

<style>
  #sim-ep0704 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0704 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0704 button:hover { background: #e8dfcf; }
  #sim-ep0704 button.sim-ep0704_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0704_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0704_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>

<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP07_04: Código LBP de una Vecindad 3&times;3</span>
  <span class="sim-ep0704_pill">P = 8, R = 1</span>
</div>

<div style="padding:16px;background:#ffffff;">

  <!-- Painel de Seleção de Exemplo -->
  <div class="sim-ep0704_panel" style="margin-bottom:14px;">
    <div style="font-size:10.5px; color:#8a8371; font-weight:600; text-align:center; margin-bottom:10px;">
      Haz clic en una celda de la vecindad para alternar entre claro y oscuro (el centro es fijo) y observa el código LBP resultante. La etiqueta p indica el índice de la ecuación.
    </div>

    <div style="display:flex; gap:6px; flex-wrap:wrap; justify-content:center;">
      <button id="sim-ep0704_b1">Ejemplo 1: Mancha Homogénea</button>
      <button id="sim-ep0704_b2" class="sim-ep0704_active">Ejemplo 2: Patrón Alternado</button>
    </div>
  </div>

  <!-- Grid Vizinhança 3x3 -->
  <div class="sim-ep0704_panel" style="margin-bottom:14px; text-align:center;">
    <div id="sim-ep0704_grid" style="display:grid; grid-template-columns:repeat(3, 60px); gap:4px; justify-content:center;"></div>
  </div>

  <!-- Painel Informativo / Debug -->
  <div id="sim-ep0704_debug" class="sim-ep0704_panel" style="font-family:monospace; font-size:11px; color:#26241d; text-align:center;">
    &ndash;
  </div>

</div>
</div>

<script>
(function(){
  function initSim07Ep04(root){
    if (!root || root.dataset.sim07Ep04Init) return;
    root.dataset.sim07Ep04Init = "1";

    var exemplos = {
      1: [[10, 10, 10], [10, 50, 10], [10, 10, 10]],
      2: [[90, 90, 90], [10, 50, 10], [90, 90, 90]]
    };

    var valores = exemplos[2].map(function(row){ return row.slice(); });
    var grid = root.querySelector('#sim-ep0704_grid');
    var dbg  = root.querySelector('#sim-ep0704_debug');
    var btn1 = root.querySelector('#sim-ep0704_b1');
    var btn2 = root.querySelector('#sim-ep0704_b2');

    var ordem = [[0, 0], [0, 1], [0, 2], [1, 2], [2, 2], [2, 1], [2, 0], [1, 0]];
    var pIndex = {};
    ordem.forEach(function(pos, p){ pIndex[pos[0] + ',' + pos[1]] = p; });
    var cenarioAtivo = 2;

    function marcarBotaoAtivo(n){
      cenarioAtivo = n;
      btn1.classList.toggle('sim-ep0704_active', n === 1);
      btn2.classList.toggle('sim-ep0704_active', n === 2);
    }

    function render(){
      grid.innerHTML = '';
      for (var r = 0; r < 3; r++){
        for (var c = 0; c < 3; c++){
          (function(r, c){
            var v = valores[r][c];
            var central = (r === 1 && c === 1);
            var div = document.createElement('div');

            var bordaCor = central ? '#26241d' : '#e4dcc8';
            var textoCor = v > 128 ? '#26241d' : '#ffffff';

            div.style.cssText = 'position:relative; height:60px; display:flex; align-items:center; justify-content:center; font-family:monospace; font-weight:700; border-radius:6px; cursor:' + (central ? 'default' : 'pointer') + '; border:2px solid ' + bordaCor + '; background:rgb(' + v + ',' + v + ',' + v + '); color:' + textoCor + '; transition:all 0.15s ease;';
            div.textContent = v;

            if (!central){
              var pLabel = document.createElement('span');
              pLabel.textContent = 'p' + pIndex[r + ',' + c];
              pLabel.style.cssText = 'position:absolute; top:2px; left:4px; font-size:9px; font-weight:400; opacity:0.8;';
              div.appendChild(pLabel);

              div.addEventListener('click', function(){
                valores[r][c] = valores[r][c] >= 128 ? 10 : 200;
                cenarioAtivo = null;
                btn1.classList.remove('sim-ep0704_active');
                btn2.classList.remove('sim-ep0704_active');
                render();
              });
            }
            grid.appendChild(div);
          })(r, c);
        }
      }

      var gc = valores[1][1];
      var bits = ordem.map(function(pos){ return valores[pos[0]][pos[1]] >= gc ? 1 : 0; });
      var lbp = 0;
      bits.forEach(function(b, p){ lbp += b * Math.pow(2, p); });

      var trans = 0;
      for (var i = 0; i < 8; i++){
        if (bits[i] !== bits[(i + 1) % 8]) trans++;
      }

      var classe = trans <= 2 ? 'UNIFORME' : 'NÃO-UNIFORME';

      if (trans <= 2) {
        dbg.style.borderColor = '#a3e4d7';
        dbg.style.background  = '#eafaf1';
        dbg.style.color       = '#04342C';
      } else {
        dbg.style.borderColor = '#f5b7b1';
        dbg.style.background  = '#fdecea';
        dbg.style.color       = '#c0392b';
      }

      dbg.textContent = 'bits (p0..p7) = ' + bits.join('') + '  |  LBP = ' + lbp + '  |  transições = ' + trans + '  |  ' + classe;
    }

    btn1.addEventListener('click', function(){
      valores = exemplos[1].map(function(row){ return row.slice(); });
      marcarBotaoAtivo(1);
      render();
    });

    btn2.addEventListener('click', function(){
      valores = exemplos[2].map(function(row){ return row.slice(); });
      marcarBotaoAtivo(2);
      render();
    });

    marcarBotaoAtivo(2);
    render();
  }

  function tryInitSim07Ep04(){
    var root = document.getElementById('sim-ep0704');
    if (root) initSim07Ep04(root); else setTimeout(tryInitSim07Ep04, 200);
  }
  tryInitSim07Ep04();
})();
</script>
""")

**Figura 7.4:** Simulador EP07_04: Código LBP de una Vecindad 3×3


In [ ]:
%%writefile EP07_04.py
# Código Python

In [ ]:
TestSuite("EP07_04.py").run()

### EP07_05 🔴 Histograma de Orientaciones de una Célula HOG

La función `hog` de `scikit-image`, empleada en el proyecto de clasificación de dígitos, divide la imagen en pequeñas **células** y, para cada una, construye un histograma de las orientaciones del gradiente ponderado por la magnitud — exactamente la etapa central descrita en la sección sobre el descriptor HOG del capítulo.

Se te ha encargado implementar este cálculo para una única célula, a partir de los valores de magnitud y orientación del gradiente **ya calculados** para cada píxel de la célula (prescindiendo del cálculo de las derivadas parciales).

#### 📋 Directrices de Implementación

1. **Dimensiones:** Leer los enteros $n$ (la célula tiene $n \times n$ píxeles) y $B$ (número de compartimentos del histograma).
2. **Magnitudes:** Leer $n$ líneas con $n$ valores reales cada una, que representan $|\nabla f(x,y)|$ para cada píxel de la célula.
3. **Orientaciones:** Leer otras $n$ líneas con $n$ valores reales cada una, que representan $\theta(x,y)$ en **grados**, ya convertidos al intervalo **no signado** $[0^\circ, 180^\circ)$, como se utiliza convencionalmente en HOG.
4. **Compartimentos:** Los $B$ compartimentos cubren $[0^\circ, 180^\circ)$ en bandas iguales de ancho $180/B$ grados. Un píxel con orientación $\theta$ pertenece al compartimento $\lfloor \theta / (180/B) \rfloor$; si ese índice es igual a $B$ (posible solo cuando $\theta$ es exactamente $180^\circ$, lo cual no debería ocurrir según la directriz del punto 3), utiliza el compartimento $B-1$.
5. **Histograma bruto:** Para cada píxel, acumula su **magnitud** (no su conteo) en el compartimento correspondiente:
$$
H[b] = \sum_{(x,y)\, :\, \text{bin}(\theta(x,y)) = b} |\nabla f(x,y)|.
$$
6. **Normalización L2:** Tras construir $H$, normalízalo para obtener $\hat H$:
$$
\hat H[b] = \frac{H[b]}{\sqrt{\sum_{j=0}^{B-1} H[j]^2 + \epsilon}}, \qquad \epsilon = 10^{-6}.
$$
7. **Salida:** Imprimir el histograma bruto $H$ (redondeado a 2 decimales) en una línea, seguido del histograma normalizado $\hat H$ (redondeado a 4 decimales) en otra línea, ambos con los $B$ valores separados por espacios, en el orden de los compartimentos.

#### 📌 Restricciones Computacionales

* ***Binning* no signado:** el intervalo de orientaciones es $[0,180)$, no $[0,360)$ — los gradientes en direcciones opuestas (diferencia de $180^\circ$) contribuyen al **mismo** compartimento, convención estándar de HOG para detección de objetos.
* **Acumulación por magnitud, no por conteo:** el histograma pondera cada píxel por su magnitud de gradiente, no simplemente cuenta cuántos píxeles caen en cada compartimento.
* **Constante de estabilización:** el $\epsilon = 10^{-6}$ en el denominador de la normalización evita la división por cero cuando la célula es completamente homogénea (todas las magnitudes nulas).

#### 📐 De dónde provienen las matrices de entrada

Antes de este EP, cada píxel $(x,y)$ de la imagen pasa por:

$$
G_x = f(x+1,y)-f(x-1,y), \qquad G_y = f(x,y+1)-f(x,y-1)
$$

$$
|\nabla f| = \sqrt{G_x^2+G_y^2}, \qquad \theta_{\text{signado}} = \operatorname{atan2}(G_y,G_x)
$$

Como HOG ignora la polaridad del contraste, el ángulo se pliega al intervalo no signado:

$$
\theta = \theta_{\text{signado}} \bmod 180°
$$

Repitiendo esto para todos los píxeles de una célula $n\times n$, se obtienen las dos matrices de entrada de este ejercicio: **magnitudes** $|\nabla f|$ y **orientaciones** $\theta \in [0°,180°)$.

#### 🧠 Fundamentación Teórica

| Etapa | Papel |
|---|---|
| Magnitud del gradiente | Pondera la contribución de cada píxel — los bordes fuertes pesan más que el ruido débil |
| Orientación no signada | Hace que el descriptor sea invariante a la polaridad del contraste (claro→oscuro vs. oscuro→claro) |
| Histograma por célula | Resume la distribución local de bordes en un vector compacto |
| Normalización L2 | Reduce la sensibilidad del descriptor a variaciones globales de iluminación y contraste |

La concatenación de los histogramas normalizados de todas las células de la imagen — no implementada en este ejercicio — forma el vector de características HOG completo, utilizado como entrada del clasificador k-NN en el proyecto del capítulo.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: Enteros $n$ y $B$.
* Siguientes $n$ líneas: $n$ magnitudes reales cada una.
* Siguientes $n$ líneas: $n$ orientaciones reales (grados, $[0,180)$) cada una.

**Salida:**

* Línea 1: los $B$ valores del histograma bruto, redondeados a 2 decimales.
* Línea 2: los $B$ valores del histograma normalizado, redondeados a 4 decimales.

#### 📌 Ejemplos

| Entrada | Salida | Observación |
|---|---|---|
| 2 2<br>1.0 2.0<br>3.0 4.0<br>10 100<br>170 20 | 5.00 5.00<br>0.7071 0.7071 | Bin de ancho 90°: $[0,90)$ y $[90,180)$; magnitudes 1 y 4 caen en el bin 0, 2 y 3 en el bin 1. |
| 2 4<br>0.0 0.0<br>0.0 0.0<br>0 0<br>0 0 | 0.00 0.00 0.00 0.00<br>0.0000 0.0000 0.0000 0.0000 | Célula homogénea: $\epsilon$ evita división por cero. |

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0705" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">  
<style>
  #sim-ep0705 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0705 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0705 button:hover { background: #e8dfcf; }
  #sim-ep0705 button.sim-ep0705_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0705_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0705_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP07_05: Histograma de Orientaciones de una Célula</span>
  <span class="sim-ep0705_pill">🔴 célula 3×3 fija</span>
</div>

  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Ajusta B y observa cómo la <b>matriz de orientaciones</b> (independiente de la de magnitudes) se mapea
      a los compartimentos mediante <code>bin = floor(θ / (180/B))</code>, y cómo las magnitudes se suman en cada bin.
    </p>

    <!-- Controle B -->
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:20px;margin-bottom:18px;">
      <div style="display:flex;justify-content:space-between;margin-bottom:8px;">
        <label style="font-size:12px;font-weight:bold;color:#2980b9;">Número de compartimentos (B)</label>
        <span id="ep0705_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">2</span>
      </div>
      <input id="ep0705_sl" style="width:100%;accent-color:#2980b9;" max="6" min="2" step="1" type="range" value="2">
    </div>

    <!-- Entrada bruta (formato VPL) -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📄 Entrada (exactamente como el programa lee por stdin)</div>
      <pre id="ep0705_stdin" style="background:#1e1e1e;color:#d4d4d4;border-radius:8px;padding:12px 14px;font-size:12px;line-height:1.5;overflow-x:auto;margin:0;"></pre>
    </div>

    <!-- Duas matrizes separadas -->
    <div style="display:flex;gap:16px;flex-wrap:wrap;margin-bottom:20px;">
      <div style="flex:1;min-width:220px;">
        <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🔢 Matriz de magnitudes |∇f|</div>
        <div id="ep0705_mag_grid" style="display:grid;grid-template-columns:repeat(3,1fr);gap:6px;"></div>
      </div>
      <div style="flex:1;min-width:220px;">
        <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📐 Matriz de orientaciones θ (grados) — coloreada por el bin</div>
        <div id="ep0705_ang_grid" style="display:grid;grid-template-columns:repeat(3,1fr);gap:6px;"></div>
      </div>
    </div>

    <!-- Regua 0-180 -->
    <div style="margin-bottom:22px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:10px;">📏 Dónde cae cada θ en la regla [0°, 180°) — <code>bin = floor(θ / ancho)</code></div>
      <div style="position:relative;height:70px;margin:0 6px;">
        <div id="ep0705_regua" style="position:absolute;top:28px;left:0;right:0;height:14px;border-radius:7px;overflow:hidden;display:flex;border:1px solid #d1d5db;"></div>
        <div id="ep0705_regua_ticks" style="position:absolute;top:44px;left:0;right:0;height:14px;"></div>
        <div id="ep0705_regua_marcas" style="position:absolute;top:0;left:0;right:0;height:26px;"></div>
      </div>
    </div>

    <!-- Faixas dos compartimentos -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📊 Rango de cada compartimento (ancho = 180° / B)</div>
      <div id="ep0705_faixas" style="display:flex;flex-wrap:wrap;gap:6px;"></div>
    </div>

    <!-- Grade de pixels colorida por bin (mag + ang juntos) -->
    <div style="margin-bottom:18px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🧩 Cada píxel: magnitud + orientación → bin</div>
      <div id="ep0705_pixels" style="display:grid;grid-template-columns:repeat(3,1fr);gap:8px;"></div>
    </div>

    <!-- Botões -->
    <div style="display:flex;gap:8px;justify-content:center;margin-bottom:14px;">
      <button id="ep0705_btn_raw" class="ep0705_btn">Histograma bruto (H)</button>
      <button id="ep0705_btn_norm" class="ep0705_btn">Histograma normalizado (Ĥ)</button>
    </div>

    <!-- Barras -->
    <div id="ep0705_bars" style="display:flex;gap:6px;align-items:flex-end;height:120px;justify-content:center;margin-bottom:14px;"></div>

    <!-- Passo a passo -->
    <div style="margin-bottom:6px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🧮 Cálculo paso a paso (floor de la división + suma de magnitudes por bin)</div>
      <div id="ep0705_passos" style="background:#f3f4f6;border-radius:8px;padding:10px 12px;font-family:monospace;font-size:11px;color:#374151;line-height:1.8;"></div>
    </div>

    <div id="ep0705_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;text-align:center;margin-top:12px;"></div>
  </div>
</div>
<style>
  #sim-ep0705 .ep0705_btn { font-size:11px;padding:6px 10px;border-radius:6px;border:1px solid #ddd;background:#fff;cursor:pointer; }
  #sim-ep0705 .ep0705_btn.ativo { background:#2980b9;color:#fff;border-color:#2980b9; }
</style>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var n = 3;
    var mags = [[1.0,2.0,0.5],[3.0,4.0,1.5],[0.8,2.5,3.2]];
    var angs = [[10,100,45],[170,20,95],[60,150,5]];
    var CORES = ["#6366f1","#0ea5e9","#10b981","#f59e0b","#ef4444","#a855f7"];

    var slEl = root.querySelector("#ep0705_sl");
    var vlEl = root.querySelector("#ep0705_vl");
    var stdinEl = root.querySelector("#ep0705_stdin");
    var magGridEl = root.querySelector("#ep0705_mag_grid");
    var angGridEl = root.querySelector("#ep0705_ang_grid");
    var reguaEl = root.querySelector("#ep0705_regua");
    var reguaTicksEl = root.querySelector("#ep0705_regua_ticks");
    var reguaMarcasEl = root.querySelector("#ep0705_regua_marcas");
    var faixasEl = root.querySelector("#ep0705_faixas");
    var pxEl = root.querySelector("#ep0705_pixels");
    var bars = root.querySelector("#ep0705_bars");
    var passosEl = root.querySelector("#ep0705_passos");
    var dbg = root.querySelector("#ep0705_debug");
    var btnRaw = root.querySelector("#ep0705_btn_raw");
    var btnNorm = root.querySelector("#ep0705_btn_norm");
    var modoNorm = false;

    function render(){
      btnRaw.classList.toggle("ativo", !modoNorm);
      btnNorm.classList.toggle("ativo", modoNorm);

      var B = parseInt(slEl.value);
      vlEl.textContent = B;
      var largura = 180/B;

      // ---- Entrada bruta (stdin) ----
      var linhas = [];
      linhas.push(n + " " + B);
      mags.forEach(function(row){ linhas.push(row.map(function(v){return v.toFixed(1);}).join(" ")); });
      angs.forEach(function(row){ linhas.push(row.join(" ")); });
      stdinEl.textContent = linhas.join("\\n");

      // ---- bin de cada pixel (floor(theta/largura), clip) ----
      var binsMat = [];
      for(var i=0;i<n;i++){
        binsMat.push([]);
        for(var j=0;j<n;j++){
          var raw = angs[i][j]/largura;
          var b = Math.floor(raw);
          if(b > B-1) b = B-1;
          if(b < 0) b = 0;
          binsMat[i].push(b);
        }
      }

      // ---- Matriz de magnitudes (grid simples) ----
      magGridEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var d = document.createElement("div");
          d.style.cssText = "text-align:center;border-radius:8px;padding:8px 4px;font-size:12px;font-family:monospace;background:#f9fafb;border:1px solid #e5e7eb;color:#374151;";
          d.textContent = mags[i][j].toFixed(1);
          magGridEl.appendChild(d);
        }
      }

      // ---- Matriz de orientações (colorida por bin, com floor explícito) ----
      angGridEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var b2 = binsMat[i][j];
          var cor2 = CORES[b2];
          var raw2 = angs[i][j]/largura;
          var d2 = document.createElement("div");
          d2.style.cssText = "text-align:center;border-radius:8px;padding:6px 4px;font-size:11px;font-family:monospace;background:"+cor2+"22;border:2px solid "+cor2+";color:#374151;";
          d2.innerHTML = "<div style=\\"font-weight:700;\\">"+angs[i][j]+"°</div>"+
            "<div style=\\"font-size:9px;color:#6b7280;\\">÷"+largura.toFixed(1)+"="+raw2.toFixed(2)+"</div>"+
            "<div style=\\"font-size:9px;font-weight:700;color:"+cor2+";\\">⌊·⌋=bin "+b2+"</div>";
          angGridEl.appendChild(d2);
        }
      }

      // ---- Régua 0-180 com faixas coloridas ----
      reguaEl.innerHTML = "";
      for(var b3=0;b3<B;b3++){
        var seg = document.createElement("div");
        seg.style.cssText = "flex:1;background:"+CORES[b3]+";opacity:0.35;border-right:1px solid rgba(255,255,255,0.6);";
        reguaEl.appendChild(seg);
      }
      // ticks (limites dos bins)
      reguaTicksEl.innerHTML = "";
      for(var b4=0;b4<=B;b4++){
        var pct = (b4*largura/180*100);
        var tick = document.createElement("div");
        tick.style.cssText = "position:absolute;left:"+pct+"%;top:0;font-size:9px;color:#6b7280;transform:translateX(-50%);white-space:nowrap;";
        tick.textContent = (b4*largura).toFixed(0)+"°";
        reguaTicksEl.appendChild(tick);
      }
      // marcadores dos angulos de cada pixel
      reguaMarcasEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var ang = angs[i][j];
          var b5 = binsMat[i][j];
          var pctm = (ang/180*100);
          var marker = document.createElement("div");
          marker.style.cssText = "position:absolute;left:"+pctm+"%;top:0;transform:translateX(-50%);display:flex;flex-direction:column;align-items:center;";
          marker.innerHTML = "<div style=\\"font-size:9px;color:"+CORES[b5]+";font-weight:700;\\">("+i+","+j+")</div>"+
            "<div style=\\"width:0;height:0;border-left:5px solid transparent;border-right:5px solid transparent;border-top:8px solid "+CORES[b5]+";\\"></div>";
          reguaMarcasEl.appendChild(marker);
        }
      }

      // ---- Faixas dos bins (legenda) ----
      faixasEl.innerHTML = "";
      for(var b=0;b<B;b++){
        var lo = (b*largura).toFixed(1);
        var hi = ((b+1)*largura).toFixed(1);
        var chip = document.createElement("div");
        chip.style.cssText = "display:flex;align-items:center;gap:6px;background:#f9fafb;border:1px solid #e5e7eb;border-radius:20px;padding:4px 10px;font-size:11px;color:#374151;";
        chip.innerHTML = "<span style=\\"width:10px;height:10px;border-radius:50%;background:"+CORES[b]+";display:inline-block;\\"></span>bin "+b+": ["+lo+"°, "+hi+"°)";
        faixasEl.appendChild(chip);
      }

      // ---- Atribuição por pixel + histograma bruto ----
      var H = new Array(B).fill(0);
      var binsPorPixel = [];
      pxEl.innerHTML = "";
      for(var i=0;i<n;i++){
        for(var j=0;j<n;j++){
          var mag = mags[i][j], ang = angs[i][j];
          var bin = binsMat[i][j];
          binsPorPixel.push({i:i, j:j, mag:mag, ang:ang, bin:bin});
          H[bin] += mag;

          var div = document.createElement("div");
          var cor = CORES[bin];
          div.style.cssText = "text-align:center;border-radius:10px;padding:8px 6px;font-size:11px;background:"+cor+"22;border:2px solid "+cor+";color:#374151;";
          div.innerHTML = "<div style=\\"font-weight:700;\\">mag="+mag.toFixed(1)+"</div>"+
            "<div style=\\"font-family:monospace;\\">θ="+ang+"°</div>"+
            "<div style=\\"font-weight:700;color:"+cor+";\\">→ bin "+bin+"</div>";
          pxEl.appendChild(div);
        }
      }

      var denom = Math.sqrt(H.reduce(function(s,v){return s+v*v;},0) + 1e-6);
      var Hn = H.map(function(v){ return v/denom; });

      // ---- Barras (coloridas por bin) ----
      var dados = modoNorm ? Hn : H;
      var maxD = Math.max.apply(null, dados.concat([0.001]));
      bars.innerHTML = "";
      dados.forEach(function(v, b){
        var col = document.createElement("div");
        col.style.cssText = "display:flex;flex-direction:column;align-items:center;gap:4px;";
        var barra = document.createElement("div");
        var altura = Math.round((v/maxD)*90) + 4;
        barra.style.cssText = "width:34px;height:"+altura+"px;background:"+CORES[b]+";border-radius:4px 4px 0 0;";
        var label = document.createElement("div");
        label.style.cssText = "font-family:monospace;font-size:10px;color:#4b5563;";
        label.textContent = modoNorm ? v.toFixed(4) : v.toFixed(2);
        var binLabel = document.createElement("div");
        binLabel.style.cssText = "font-size:9px;color:#9ca3af;";
        binLabel.textContent = "bin "+b;
        col.appendChild(barra);
        col.appendChild(label);
        col.appendChild(binLabel);
        bars.appendChild(col);
      });

      // ---- Passo a passo (floor + soma) ----
      var passos = [];
      for(var b=0;b<B;b++){
        var contribs = binsPorPixel.filter(function(p){ return p.bin===b; });
        var termos = contribs.map(function(p){ return p.mag.toFixed(2)+" (θ="+p.ang+"°→⌊"+(p.ang/largura).toFixed(2)+"⌋="+p.bin+")"; }).join(" + ");
        if(termos === "") termos = "(nenhum pixel)";
        passos.push("<span style=\\"color:"+CORES[b]+";font-weight:700;\\">H["+b+"]</span> = "+termos+" = <b>"+H[b].toFixed(2)+"</b>");
      }
      passosEl.innerHTML = passos.join("<br>");

      dbg.textContent = "H=[" + H.map(function(v){return v.toFixed(2);}).join(", ") + "]  |  Ĥ=[" +
        Hn.map(function(v){return v.toFixed(4);}).join(", ") + "]";
    }

    slEl.addEventListener("input", render);
    btnRaw.addEventListener("click", function(){ modoNorm = false; render(); });
    btnNorm.addEventListener("click", function(){ modoNorm = true; render(); });
    render();
  }
  function tryInit(){
    var root = document.getElementById("sim-ep0705");
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 7.5:** Simulador EP07_05: Histograma HOG de una Celda (mapeo de ángulos a bins)


In [ ]:
%%writefile EP07_05.py
# Código Python

In [ ]:
TestSuite("EP07_05.py").run()

### EP07_06 🟣 *Pipeline* Completo: Descriptores + k-NN + Evaluación Multiclase

Este ejercicio integra las tres etapas centrales del capítulo en un único *pipeline*, reproduciendo en miniatura el **Proyecto Práctico 2** (clasificación de texturas sintéticas por LBP): un conjunto de histogramas de descriptores **ya extraídos** (como si fueran histogramas LBP) se utiliza para entrenar un clasificador k-NN, que a su vez se evalúa sobre un conjunto de prueba independiente mediante una matriz de confusión multiclase.

A diferencia de EP07_01, aquí el espacio de características tiene dimensión arbitraria $H$ (el tamaño del histograma), existen más de dos clases, y la métrica de distancia es un parámetro de entrada — lo que permite reproducir el experimento de comparación de métricas discutido en el capítulo.

#### 📋 Directrices de Implementación

1. **Clases:** Leer el entero $C$ (número de clases) seguido de $C$ nombres de clase (*strings* sin espacio), en el orden en que deben aparecer en la matriz de confusión.
2. **Configuración:** Leer el entero $H$ (dimensión de los histogramas), la *string* $M$ (métrica: `euclidiana` o `manhattan`) y el entero impar $k$.
3. **Entrenamiento:** Leer el entero $N$ y, a continuación, $N$ líneas, cada una conteniendo el nombre de la clase seguido de $H$ valores reales (el histograma de descriptor).
4. **Prueba:** Leer el entero $Q$ y, a continuación, $Q$ líneas, cada una conteniendo el nombre de la clase **real** seguido de $H$ valores reales (el histograma de descriptor de la muestra de prueba).
5. **Distancia:** Para cada muestra de prueba, calcule la distancia a cada ejemplo de entrenamiento usando la métrica $M$:
$$
d_{\text{euclidiana}}(u,v) = \sqrt{\sum_{j=1}^{H}(u_j-v_j)^2}, \qquad
d_{\text{manhattan}}(u,v) = \sum_{j=1}^{H} |u_j - v_j|.
$$
6. **Clasificación k-NN:** Seleccione los $k$ ejemplos de entrenamiento más cercanos (desempate de distancia por el orden de lectura, como en EP07_01) y clasifique por la clase mayoritaria entre ellos. En caso de **empate de votación** entre dos o más clases, elija la que aparezca **primero** en la lista de clases del ítem 1.
7. **Matriz de confusión:** Construya una matriz $C \times C$ en la que la fila corresponde a la clase real y la columna a la clase prevista, siguiendo el orden de clases del ítem 1.
8. **Precisión:** Calcule la precisión global como la razón entre aciertos y $Q$.
9. **Salida:** Para cada muestra de prueba, en el orden de entrada, imprimir la clase prevista. A continuación, imprimir la matriz de confusión (una fila por clase real, valores separados por espacio, en el orden de las clases). Finalmente, imprimir la precisión redondeada a 4 decimales.

#### 📌 Restricciones Computacionales

* **Métrica seleccionable:** implemente ambas distancias; la métrica $M$ define cuál se utiliza en toda la ejecución (no es posible mezclar métricas en la misma llamada).
* **Desempate de votación determinístico:** el criterio del ítem 6 (orden de la lista de clases) debe seguirse incluso cuando el empate involucra más de dos clases.
* **Independencia de entrenamiento y prueba:** no hay necesidad de validar que las muestras de prueba no aparecen en el entrenamiento — asuma que la entrada es válida.

#### 🧠 Fundamentación Teórica

| Etapa del ejercicio | Etapa correspondiente en el capítulo |
|---|---|
| Histogramas de entrenamiento/prueba ya extraídos | `descritor_lbp` aplicado a las texturas sintéticas |
| Distancia euclidiana o Manhattan | Parámetro `metric` del `KNeighborsClassifier` |
| Votación mayoritaria con $k$ vecinos | `KNeighborsClassifier.predict` |
| Matriz de confusión $C\times C$ | `confusion_matrix` de `scikit-learn` |
| Precisión global | `accuracy_score` de `scikit-learn` |

Este ejercicio evidencia, de forma controlada, un resultado discutido en el capítulo: la **elección de la métrica de distancia** y del **valor de $k$** puede alterar la clase prevista para una misma muestra, incluso manteniendo fijo el descriptor utilizado — reforzando que, en el reconocimiento de patrones clásico, el descriptor, la métrica y el clasificador forman un sistema interdependiente, y no piezas aisladas.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

* Línea 1: entero $C$ seguido de $C$ nombres de clase.
* Línea 2: entero $H$, *string* $M$ y entero $k$.
* Línea 3: entero $N$.
* Siguientes $N$ líneas de entrenamiento: nombre de la clase seguido de $H$ reales.
* Siguiente línea: entero $Q$.
* Siguientes $Q$ líneas de prueba: nombre de la clase real seguido de $H$ reales.

**Salida:**

* $Q$ líneas con la clase prevista de cada muestra de prueba, en el orden de entrada.
* $C$ líneas con la matriz de confusión (una fila por clase real).
* Última línea: `Acuracia: <valor>`.

#### 📌 Ejemplos

| Entrada (resumida) | Salida | Observación |
|---|---|---|
| 2 granular listrada<br>2 euclidiana 1<br>4<br>granular 0.9 0.1<br>granular 0.8 0.2<br>listrada 0.1 0.9<br>listrada 0.2 0.8<br>2<br>granular 0.85 0.15<br>listrada 0.15 0.85 | granular<br>listrada<br>1 0<br>0 1<br>Acuracia: 1.0000 | Con $k=1$, cada prueba se clasifica por el vecino de entrenamiento más cercano. |

> ### 📝 Nota
>
> Este simulador usa un conjunto simplificado de **3 clases** (`granular`, `listrada`, `manchada`) sobre puntos 2D fictícios, solo para ilustrar el *pipeline* de votación, desempate y matriz de confusión del k-NN. En el **EP07_07**, aplicarás esta misma lógica a un mosaico de imagen real, que introduce una cuarta clase (`xadrez`) y sustituye los puntos 2D por histogramas LBP extraídos directamente de los píxeles de la imagen.

In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0706" style="background-color:#fbf7ee;border-radius:18px;border:1px solid #e4dcc8;overflow:hidden;margin-top:20px;font-family:'Inter',system-ui,sans-serif;box-shadow:0 1px 2px rgba(38,36,29,0.04);position:relative;">  
<style>
  #sim-ep0706 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0706 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #e4dcc8; background: #f1ead7; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0706 button:hover { background: #e8dfcf; }
  #sim-ep0706 button.sim-ep0706_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0706_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #e4dcc8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0706_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP07_06: Pipeline k-NN Multiclase</span>
  <span class="sim-ep0706_pill">6 Entrenamiento &middot; 3 Prueba &middot; 3 Clases</span>
</div>

  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
      Elija la métrica, el valor de k y la muestra de prueba (★). Vea los k vecinos más cercanos, la votación,
      el desempate cuando sea necesario, y cómo esto se propaga a la matriz de confusión y la precisión del conjunto completo.
    </p>

    <!-- Controles -->
    <div style="display:flex;flex-wrap:wrap;gap:18px;justify-content:center;margin-bottom:16px;">
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Métrica (M)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_be" class="ep0706_btn">Euclidiana</button>
          <button id="ep0706_bm" class="ep0706_btn">Manhattan</button>
        </div>
      </div>
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Vecinos (k)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_k1" class="ep0706_btn">k=1</button>
          <button id="ep0706_k3" class="ep0706_btn">k=3</button>
          <button id="ep0706_k5" class="ep0706_btn">k=5</button>
        </div>
      </div>
      <div>
        <div style="font-size:10px;color:#5e5a4a;margin-bottom:4px;text-align:center;">Muestra de prueba (★)</div>
        <div style="display:flex;gap:6px;">
          <button id="ep0706_t0" class="ep0706_btn">prueba 1</button>
          <button id="ep0706_t1" class="ep0706_btn">prueba 2</button>
          <button id="ep0706_t2" class="ep0706_btn">prueba 3</button>
        </div>
      </div>
    </div>

    <!-- Legenda -->
    <div id="ep0706_legenda" style="display:flex;gap:10px;justify-content:center;margin-bottom:10px;"></div>

    <!-- Dispersao 2D -->
    <div style="max-width:340px;margin:0 auto 16px auto;height:300px;border:1px solid #e5e7eb;border-radius:12px;background:#fafafa;">
      <div id="ep0706_svg_container" style="width:100%;height:100%;"></div>
    </div>

    <!-- Distancias ordenadas -->
    <div style="margin-bottom:16px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📏 Distancias a la muestra de prueba (ordenadas) — <span style="font-weight:400;font-size:10px;color:#8a8672;">#i = orden de lectura en la lista de entrenamiento (pase el mouse)</span></div>
      <div id="ep0706_dists" style="display:grid;grid-template-columns:1fr 1fr;gap:2px 10px;font-family:monospace;font-size:10px;"></div>
    </div>

    <!-- Votacao -->
    <div style="margin-bottom:16px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">🗳️ Votación entre los k vecinos</div>
      <div id="ep0706_votos" style="display:flex;gap:10px;justify-content:center;margin-bottom:6px;"></div>
      <div id="ep0706_previsao" style="text-align:center;font-size:12px;font-weight:bold;"></div>
    </div>

    <!-- Matriz de confusao + acuracia (conjunto de teste inteiro) -->
    <div style="margin-bottom:8px;">
      <div style="font-size:12px;font-weight:bold;color:#5e5a4a;margin-bottom:6px;">📋 Matriz de confusión y precisión — ejecutando el pipeline sobre las 3 muestras de prueba</div>
      <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:center;justify-content:center;">
        <table id="ep0706_cm" style="border-collapse:collapse;font-size:11px;font-family:monospace;"></table>
        <div id="ep0706_acc" style="font-size:13px;font-weight:bold;color:#5e5a4a;"></div>
      </div>
    </div>

    <div id="ep0706_debug" style="background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:10px;color:#1565c0;text-align:center;margin-top:12px;"></div>
  </div>
</div>
<style>
  #sim-ep0706 .ep0706_btn { font-size:11px;padding:5px 10px;border-radius:6px;border:1px solid #ddd;background:#fff;cursor:pointer; }
  #sim-ep0706 .ep0706_btn.ativo { background:#7c3aed;color:#fff;border-color:#7c3aed; }
</style>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var classes = ["granular","listrada","manchada"];
    var CORES = {granular:"#6366f1", listrada:"#f59e0b", manchada:"#10b981"};

    var trainPts = [
      {nome:"granular_1", cls:"granular", x:0.70, y:0.70},
      {nome:"granular_2", cls:"granular", x:0.25, y:0.85},
      {nome:"listrada_1", cls:"listrada", x:0.85, y:0.50},
      {nome:"listrada_2", cls:"listrada", x:0.60, y:0.15},
      {nome:"manchada_1", cls:"manchada", x:0.30, y:0.30},
      {nome:"manchada_2", cls:"manchada", x:0.15, y:0.55}
    ];
    var testPts = [
      {nome:"teste 1", cls:"granular", x:0.50, y:0.50},
      {nome:"teste 2", cls:"listrada", x:0.70, y:0.20},
      {nome:"teste 3", cls:"manchada", x:0.20, y:0.40}
    ];

    var svgContainer = root.querySelector("#ep0706_svg_container");
    var svg = svgNS("svg");
    svg.setAttribute("viewBox", "0 0 100 100");
    svg.setAttribute("style", "width:100%;height:100%;");
    svgContainer.appendChild(svg);
    var legendaEl = root.querySelector("#ep0706_legenda");
    var distsEl = root.querySelector("#ep0706_dists");
    var votosEl = root.querySelector("#ep0706_votos");
    var previsaoEl = root.querySelector("#ep0706_previsao");
    var cmEl = root.querySelector("#ep0706_cm");
    var accEl = root.querySelector("#ep0706_acc");
    var dbg = root.querySelector("#ep0706_debug");

    var be = root.querySelector("#ep0706_be"), bm = root.querySelector("#ep0706_bm");
    var bk1 = root.querySelector("#ep0706_k1"), bk3 = root.querySelector("#ep0706_k3"), bk5 = root.querySelector("#ep0706_k5");
    var bt0 = root.querySelector("#ep0706_t0"), bt1 = root.querySelector("#ep0706_t1"), bt2 = root.querySelector("#ep0706_t2");

    var metrica = "euclidiana", k = 1, testSel = 0;

    function dist(u, v){
      var dx = u.x-v.x, dy = u.y-v.y;
      if(metrica === "euclidiana") return Math.sqrt(dx*dx+dy*dy);
      return Math.abs(dx)+Math.abs(dy);
    }

    function knnPredict(xtest){
      var ds = trainPts.map(function(p, i){ return {i:i, p:p, d:dist(xtest, p)}; });
      ds.sort(function(a,b){ return a.d - b.d; }); // ordem estavel = desempate por ordem de leitura
      var viz = ds.slice(0, k);
      var votos = {}; classes.forEach(function(c){ votos[c]=0; });
      viz.forEach(function(v){ votos[v.p.cls]++; });
      var maxV = Math.max.apply(null, classes.map(function(c){return votos[c];}));
      var empatados = classes.filter(function(c){ return votos[c]===maxV; });
      var pred = empatados[0]; // primeira classe da lista entre as empatadas
      return {pred:pred, viz:viz, votos:votos, empatados:empatados, ordenados:ds};
    }

    function svgNS(tag){
      // Concatenado de propósito: evita que filtros de auto-link do Moodle
      // reconheçam "http://www.w3.org/2000/svg" como URL e insiram uma tag <a>
      // dentro desta string, o que quebraria a sintaxe do createElementNS.
      var SVG_NS = "http" + "://www.w3.org/2000/svg";
      return document.createElementNS(SVG_NS, tag);
    }

    function render(){
      be.classList.toggle("ativo", metrica==="euclidiana");
      bm.classList.toggle("ativo", metrica==="manhattan");
      bk1.classList.toggle("ativo", k===1);
      bk3.classList.toggle("ativo", k===3);
      bk5.classList.toggle("ativo", k===5);
      bt0.classList.toggle("ativo", testSel===0);
      bt1.classList.toggle("ativo", testSel===1);
      bt2.classList.toggle("ativo", testSel===2);

      // Legenda
      legendaEl.innerHTML = "";
      classes.forEach(function(c){
        var chip = document.createElement("div");
        chip.style.cssText = "display:flex;align-items:center;gap:5px;font-size:11px;color:#374151;";
        chip.innerHTML = '<span style="width:10px;height:10px;border-radius:50%;background:'+CORES[c]+';display:inline-block;"></span>'+c;
        legendaEl.appendChild(chip);
      });

      var xt = testPts[testSel];
      var r = knnPredict(xt);
      var vizIdx = r.viz.map(function(v){ return v.i; });

      // ---- SVG: pontos de treino, linhas para vizinhos, estrela de teste ----
      svg.innerHTML = "";
      // grade leve
      for(var g=1; g<4; g++){
        var lineV = svgNS("line");
        lineV.setAttribute("x1", g*25); lineV.setAttribute("y1", 0);
        lineV.setAttribute("x2", g*25); lineV.setAttribute("y2", 100);
        lineV.setAttribute("stroke", "#eee"); lineV.setAttribute("stroke-width", "0.4");
        svg.appendChild(lineV);
        var lineH = svgNS("line");
        lineH.setAttribute("x1", 0); lineH.setAttribute("y1", g*25);
        lineH.setAttribute("x2", 100); lineH.setAttribute("y2", g*25);
        lineH.setAttribute("stroke", "#eee"); lineH.setAttribute("stroke-width", "0.4");
        svg.appendChild(lineH);
      }
      // linhas ate os vizinhos (desenhadas antes dos pontos, para ficarem por baixo)
      vizIdx.forEach(function(i){
        var p = trainPts[i];
        var line = svgNS("line");
        line.setAttribute("x1", xt.x*100); line.setAttribute("y1", (1-xt.y)*100);
        line.setAttribute("x2", p.x*100); line.setAttribute("y2", (1-p.y)*100);
        line.setAttribute("stroke", CORES[p.cls]); line.setAttribute("stroke-width", "0.6");
        line.setAttribute("stroke-dasharray", "1.5,1"); line.setAttribute("opacity", "0.7");
        svg.appendChild(line);
      });
      // pontos de treino
      trainPts.forEach(function(p, i){
        var isViz = vizIdx.indexOf(i) !== -1;
        if(isViz){
          var halo = svgNS("circle");
          halo.setAttribute("cx", p.x*100); halo.setAttribute("cy", (1-p.y)*100);
          halo.setAttribute("r", 5); halo.setAttribute("fill", "none");
          halo.setAttribute("stroke", CORES[p.cls]); halo.setAttribute("stroke-width", "0.8");
          svg.appendChild(halo);
        }
        var c = svgNS("circle");
        c.setAttribute("cx", p.x*100); c.setAttribute("cy", (1-p.y)*100);
        c.setAttribute("r", 3.2);
        c.setAttribute("fill", CORES[p.cls]);
        c.setAttribute("stroke", "#fff"); c.setAttribute("stroke-width", "0.6");
        c.setAttribute("opacity", isViz ? "1" : "0.55");
        svg.appendChild(c);
      });
      // estrela de teste
      var correto = (r.pred === xt.cls);
      var estCor = correto ? "#16a34a" : "#dc2626";
      var halo2 = svgNS("circle");
      halo2.setAttribute("cx", xt.x*100); halo2.setAttribute("cy", (1-xt.y)*100);
      halo2.setAttribute("r", 5.5); halo2.setAttribute("fill", "#fff");
      halo2.setAttribute("stroke", estCor); halo2.setAttribute("stroke-width", "0.8");
      svg.appendChild(halo2);
      var txt = svgNS("text");
      txt.setAttribute("x", xt.x*100); txt.setAttribute("y", (1-xt.y)*100+1.8);
      txt.setAttribute("text-anchor", "middle"); txt.setAttribute("font-size", "6.5");
      txt.setAttribute("fill", estCor);
      txt.textContent = "★";
      svg.appendChild(txt);

      // ---- Distancias ordenadas ----
      distsEl.innerHTML = "";
r.ordenados.forEach(function(v, ord){
  var dentroK = ord < k;
  var row = document.createElement("div");
  row.style.cssText = "display:flex;justify-content:space-between;align-items:center;padding:2px 6px;border-radius:6px;" +
    (dentroK ? "background:"+CORES[v.p.cls]+"22;border:1px solid "+CORES[v.p.cls]+";" : "background:#f9fafb;border:1px solid #f1f1f1;color:#9ca3af;");
  row.innerHTML =
    '<span style="display:flex;align-items:center;gap:4px;">' +
      (dentroK ? '✓' : '\u00A0') +
      '<span title="posición de lectura en la lista original de entrenamiento — usada para desempate cuando dos distancias son iguales" ' +
        'style="background:#eee;color:#9ca3af;border-radius:3px;padding:0 3px;font-size:8.5px;cursor:help;">#' + (v.i+1) + '</span>' +
      ' ' + v.p.nome + ' <span style="color:'+CORES[v.p.cls]+';font-weight:700;">('+v.p.cls+')</span>' +
    '</span>' +
    '<span>d='+v.d.toFixed(4)+'</span>';
  distsEl.appendChild(row);
});

      // ---- Votacao ----
      votosEl.innerHTML = "";
      classes.forEach(function(c){
        var venceu = (c === r.pred);
        var empatou = r.empatados.length > 1 && r.empatados.indexOf(c) !== -1;
        var div = document.createElement("div");
        div.style.cssText = "text-align:center;border-radius:10px;padding:8px 14px;font-size:12px;" +
          (venceu ? "background:"+CORES[c]+"22;border:2px solid "+CORES[c]+";" : "background:#f9fafb;border:1px solid #e5e7eb;color:#9ca3af;");
        div.innerHTML = '<div style="font-weight:700;color:'+CORES[c]+';">'+c+'</div><div style="font-size:16px;font-weight:700;">'+r.votos[c]+'</div>' +
          (empatou ? '<div style="font-size:9px;color:#b91c1c;">empate</div>' : '');
        votosEl.appendChild(div);
      });
      var msgEmpate = r.empatados.length > 1 ? " (empate entre "+r.empatados.join(", ")+" — desempate pela ordem da lista de classes)" : "";
      previsaoEl.innerHTML = 'Classe prevista: <span style="color:'+CORES[r.pred]+';">'+r.pred+'</span>' + msgEmpate +
        ' &nbsp;|&nbsp; classe real: <span style="color:'+CORES[xt.cls]+';">'+xt.cls+'</span> ' + (correto ? '✅' : '❌');

      // ---- Matriz de confusao + acuracia sobre as 3 amostras de teste ----
      var cm = [[0,0,0],[0,0,0],[0,0,0]];
      var acertos = 0;
      var predsGlobais = [];
      testPts.forEach(function(tp){
        var rr = knnPredict(tp);
        predsGlobais.push(rr.pred);
        var iReal = classes.indexOf(tp.cls);
        var iPrev = classes.indexOf(rr.pred);
        cm[iReal][iPrev]++;
        if(rr.pred === tp.cls) acertos++;
      });
      var acc = acertos/testPts.length;

      var thead = '<tr><td></td>' + classes.map(function(c){ return '<td style="padding:4px 8px;color:'+CORES[c]+';font-weight:700;">'+c.slice(0,4)+'</td>'; }).join('') + '</tr>';
      var rows = classes.map(function(cReal, i){
        var cells = classes.map(function(cPrev, j){
          var v = cm[i][j];
          var diag = (i===j);
          var bg = v===0 ? '#fff' : (diag ? '#dcfce7' : '#fee2e2');
          return '<td style="padding:4px 10px;text-align:center;border:1px solid #e5e7eb;background:'+bg+';">'+v+'</td>';
        }).join('');
        return '<tr><td style="padding:4px 8px;color:'+CORES[cReal]+';font-weight:700;">'+cReal.slice(0,4)+'</td>'+cells+'</tr>';
      }).join('');
      cmEl.innerHTML = thead + rows;
      accEl.textContent = "Precisión: " + acc.toFixed(4) + " (" + acertos + "/" + testPts.length + ")";

      dbg.textContent = "M="+metrica+" k="+k+" | teste_sel="+xt.nome+" | y_pred(todas)=["+predsGlobais.join(", ")+"]";
    }

    be.addEventListener("click", function(){ metrica="euclidiana"; render(); });
    bm.addEventListener("click", function(){ metrica="manhattan"; render(); });
    bk1.addEventListener("click", function(){ k=1; render(); });
    bk3.addEventListener("click", function(){ k=3; render(); });
    bk5.addEventListener("click", function(){ k=5; render(); });
    bt0.addEventListener("click", function(){ testSel=0; render(); });
    bt1.addEventListener("click", function(){ testSel=1; render(); });
    bt2.addEventListener("click", function(){ testSel=2; render(); });

    render();
  }
  function tryInit(){
    var root = document.getElementById("sim-ep0706");
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 7.6:** Simulador EP07_06: *Pipeline* k-NN Multi-Clase (votación, desempate y matriz de confusión)


In [ ]:
%%writefile EP07_06.py
# Código Python

In [ ]:
TestSuite("EP07_06.py").run()

### EP07_07 ⚫ Clasificación Real de un Mosaico de Texturas mediante LBP + k-NN

En los ejercicios anteriores, el descriptor LBP (**EP07_04**) y el clasificador k-NN multiclase (**EP07_06**) se estudiaron por separado, siempre a partir de datos ya proporcionados en la entrada: vecindades $3\times3$ aisladas o histogramas previamente extraídos. En este ejercicio de cierre del capítulo, el programa deberá **leer una imagen real**, en formato **PGM ASCII (P2)**, calcular el descriptor LBP directamente a partir de los píxeles y, a continuación, clasificar cada región mediante el k-NN, reproduciendo, a escala reducida, el flujo completo de un sistema de reconocimiento de texturas. Este enfoque también anticipa la idea de **clasificación por mosaico de regiones**, relacionada con la segmentación semántica estudiada en un capítulo posterior.

El simulador interactivo del **EP07_06** utilizaba solo tres clases (`granular`, `listrada` y `manchada`) representadas por puntos bidimensionales ficticios. En este ejercicio, se añade una cuarta clase, **xadrez**, y los puntos se sustituyen por histogramas LBP extraídos de una imagen real.

La imagen de entrada es un **mosaico** formado por una cuadrícula $G\times G$ de bloques cuadrados de $S\times S$ píxeles. Cada bloque contiene una muestra de una de las cuatro clases de textura sintética del capítulo: **granular**, **listrada**, **manchada** o **xadrez** (patrón de tablero con intensidades alternadas). Como en los demás ejercicios del libro, la carga de la imagen se realiza mediante la función didáctica `mm.readImg`.

> ### 💡 ¿Por qué un mosaico único y no varias imágenes?
>
> La entrada reúne las $G \times G$ muestras de textura en un único archivo **PGM**, solo para simplificar la lectura de los datos y evitar la apertura de varios archivos. Para el algoritmo, esto no altera el procesamiento: cada bloque se trata de forma independiente, como si fuera una imagen aislada.
> La única excepción es la **exclusión del borde** (ítem 4 a continuación).

#### 📋 Directrices de Implementación

1. **Lectura de las dimensiones de la imagen**

   Leer, mediante la entrada estándar, dos líneas que contengan, respectivamente, el número de filas $L$ y el número de columnas $C$ del mosaico (ambos múltiplos del tamaño de bloque $S$, con $L=C$).

2. **Carga de la imagen**

   Utilizar la función didáctica

   ```python
   f = mm.readImg(L, C)
   ```

   para leer los $L \times C$ valores de intensidad (tonos de gris, `uint8`) del mosaico.

3. **Parámetros de la cuadrícula**

   Leer el entero $G$ (número de bloques por lado) y el entero $S$ (tamaño del lado de cada bloque, en píxeles), satisfaciendo $L = C = G \times S$.

4. **Cálculo del código LBP por píxel**

   Para cada píxel **interior** de la imagen (es decir, que no esté en el borde global de `f` — fila o columna $0$ o $L-1$/$C-1$), calcule el código LBP con $P=8$ vecinos y radio $R=1$, recorriendo los vecinos en sentido **horario** a partir de la esquina superior izquierda, exactamente como en el EP07_04: `[lin-1][col-1]`, `[lin-1][col]`, `[lin-1][col+1]`, `[lin][col+1]`, `[lin+1][col+1]`, `[lin+1][col]`, `[lin+1][col-1]`, `[lin][col-1]`.

   Los píxeles en el borde global de la imagen **no** poseen vecindad completa y deben ser **ignorados** (no contribuyen a ningún histograma). Esto incluye píxeles de borde que caen en el interior de un bloque (la exclusión es siempre respecto al borde de la imagen completa, no al borde de cada bloque).

5. **Histograma LBP uniforme por bloque (10 compartimentos)**

   Para cada bloque $(i,j)$ de la cuadrícula ($i,j = 0,\ldots,G-1$), acumule, entre sus píxeles válidos (ítem 4), un histograma $H^{(i,j)}$ de $10$ compartimentos:

   * Considerando la secuencia circular de bits $s_0,\ldots,s_7$ del píxel (misma regla de transiciones del EP07_04): si el número de transiciones es $\le 2$ (patrón **uniforme**), el píxel contribuye al compartimento $\operatorname{popcount}(s_0,\ldots,s_7) \in \{0,\ldots,8\}$ (número de bits iguales a `1`);
   * En caso contrario (patrón **no uniforme**), el píxel contribuye al compartimento $9$.

   Al final, normalice el histograma de cada bloque dividiendo por el número de píxeles válidos contenidos en él, obteniendo $\hat H^{(i,j)}$, con $\sum_{b=0}^{9} \hat H^{(i,j)}[b] = 1$.

6. **Prototipos de entrenamiento**

   Leer el entero $Ncl$ (número de clases) seguido de $Ncl$ nombres de clase (orden que define la matriz de confusión y el desempate de votación, como en el EP07_06); a continuación, leer la *cadena* $M$ (métrica: `euclidiana` o `manhattan`) y el entero impar $k$; por último, leer el entero $N$ (número de prototipos) y, para cada uno, el nombre de la clase seguido de $10$ valores reales (histograma prototipo ya normalizado).

7. **Clasificación k-NN de cada bloque**

   Para cada bloque, calcule la distancia de $\hat H^{(i,j)}$ a cada uno de los $N$ prototipos, usando la métrica $M$ (mismas fórmulas del EP07_06). Seleccione los $k$ prototipos más cercanos (desempate de distancia por el orden de lectura de los prototipos) y clasifique por la clase mayoritaria (desempate de votación por el orden de las clases del ítem 6).

8. **Etiquetas reales y evaluación**

   Leer, en una única línea, los $G \times G$ nombres de clase **reales** de cada bloque, en orden de lectura por fila de la cuadrícula (bloque $(0,0)$, $(0,1)$, …, $(0,G-1)$, $(1,0)$, …). Construya la matriz de confusión $Ncl \times Ncl$ (fila = clase real, columna = clase predicha) y la precisión global.

9. **Salida**

   Imprimir, para cada bloque (en el mismo orden de lectura de las etiquetas reales del ítem 8), la clase predicha. A continuación, imprimir la matriz de confusión (una fila por clase real, en el orden del ítem 6). Por último, imprimir la precisión, redondeada a 4 decimales.

#### 📌 Restricciones Computacionales

* **Descriptor fijo:** $P=8$, $R=1$ y $10$ compartimentos (según el ítem 5) son fijos en este ejercicio — no se leen de la entrada.
* **Exclusión de borde global, no de bloque:** un píxel en el límite entre dos bloques, pero en el interior de la imagen, es válido y contribuye normalmente al histograma del bloque al que pertenece.
* **Orden de lectura como criterio de desempate:** tanto el desempate de distancia (ítem 7) como el de votación (ítem 7) siguen exactamente las mismas convenciones del EP07_01 y del EP07_06.
* **Prototipos como entrada, no aprendidos:** a diferencia del Proyecto Práctico 2, los histogramas de entrenamiento se proporcionan directamente en la entrada; el programa no debe generar texturas sintéticas.

#### 🧠 Fundamentación Teórica

| Etapa del ejercicio | Etapa correspondiente en el capítulo |
|---|---|
| Lectura de la imagen mediante `mm.readImg` | Adquisición de la imagen en el *pipeline* de reconocimiento de patrones |
| Código LBP por píxel (EP07_04) | `local_binary_pattern(imagen, P=8, R=1, method="uniform")` |
| Histograma de 10 compartimentos por bloque | Función `descritor_lbp` del Proyecto Práctico 2 (`bins=10`, `range=(0, P+2)`) |
| Clasificación k-NN con métrica seleccionable (EP07_06) | `KNeighborsClassifier` entrenado sobre `X_textura` |
| Matriz de confusión $Ncl\times Ncl$ y precisión | `confusion_matrix` y `accuracy_score` sobre `yt_teste` |

Este ejercicio evidencia, con píxeles reales en lugar de valores sintéticos, una limitación discutida en la sección final del capítulo: clases de textura visualmente distintas para un observador humano — como **granular** y **manchada** — pueden producir histogramas LBP similares cuando la vecindad considerada es pequeña ($R=1$), pues ambas presentan alta frecuencia de patrones no uniformes en la escala de un único píxel. En cambio, la clase **xadrez**, por poseer bordes regulares y repetitivos, tiende a separarse con mayor facilidad. Se espera que la matriz de confusión producida refleje exactamente ese patrón de confusión parcial.

#### 📦 Especificación de Entrada y Salida (VPL)

**Entrada:**

```
L
C
[matriz L x C de la imagen]
G S
Ncl nombre_clase_1 ... nombre_clase_Ncl
M k
N
nombre_clase h0 h1 ... h9      (repetida N veces)
etiqueta(0,0) etiqueta(0,1) ... etiqueta(G-1,G-1)
```

**Salida:**

* $G \times G$ líneas con la clase predicha de cada bloque, en el orden de lectura de la cuadrícula.
* $Ncl$ líneas con la matriz de confusión (una fila por clase real, valores separados por espacio).
* Última línea: `Acuracia: <valor>`.

#### 📌 Ejemplo (verificación manual)

Para comprobar la implementación del descriptor antes de probarla sobre un mosaico completo, considere una imagen $6\times6$ **homogénea**, con todos los píxeles de intensidad $100$, tratada como un único bloque ($G=1$, $S=6$). Como todo píxel interior tiene los 8 vecinos con intensidad igual a la del centro ($g_p \ge g_c$ en todos los casos), todos los bits $s_p$ valen `1`, el número de transiciones es $0$ (uniforme) y el compartimento es $\operatorname{popcount}(11111111)=8$. El histograma del único bloque es, por tanto, `0 0 0 0 0 0 0 0 1 0`.

| Entrada (resumida) | Salida | Observación |
|---|---|---|
| 6<br>6<br>[36 valores iguales a 100]<br>1 6<br>2 uniforme outra<br>euclidiana 1<br>2<br>uniforme 0 0 0 0 0 0 0 0 1 0<br>outra 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1 0.1<br>uniforme | uniforme<br>1 0<br>0 0<br>Acuracia: 1.0000 | Distancia del bloque al prototipo `uniforme` es exactamente $0$; la clase `outra` no aparece en la etiqueta real, por eso su fila en la matriz de confusión es nula. |

#### 📌 Archivos de Referencia (.pgm)

Para depuración local, dos mosaicos de prueba en el patrón ASCII P2 están disponibles (anexados a esta entrega; al integrarlos al repositorio del capítulo, guárdelos en `all/cap07/dados/EP07/`):

* 📥 **Caso 1 — Mosaico simple (`Caso1_Mosaico_Simples.pgm`)**: cuadrícula $2\times2$ de bloques de $24\times24$ píxeles, una muestra de cada una de las cuatro clases, con bajo ruido — útil para validar la lectura de la imagen y la lógica de clasificación en un escenario controlado.
* 📥 **Caso 2 — Mosaico mixto (`Caso2_Mosaico_Misto.pgm`)**: cuadrícula $3\times3$ de bloques de $16\times16$ píxeles, con clases repetidas y mayor variabilidad — escenario en el que la confusión entre **granular** y **manchada** discutida en la Fundamentación Teórica tiende a manifestarse.

La [Figura 7.7](#fig-07-ep07) exhibe los dos mosaicos, para inspección visual antes de la implementación.

In [ ]:
import os
import urllib.request
import numpy as np

def garantir_e_baixar_arquivo(nome_arquivo):
    diretorio_local = "dados/EP07"
    caminho_local = os.path.join(diretorio_local, nome_arquivo)
    
    # Crear el directorio local si no existe
    if not os.path.exists(diretorio_local):
        os.makedirs(diretorio_local)
        
    # Si el archivo no existe localmente, se descarga del repositorio remoto
    if not os.path.exists(caminho_local):
        url_base = "https://raw.githubusercontent.com/fzampirolli/"
        url_base += "pdi-vc/master/all/cap07/dados/EP07"
        url_arquivo = f"{url_base}/{nome_arquivo}"
        print(f"Descargando {nome_arquivo} desde GitHub...")
        try:
            urllib.request.urlretrieve(url_arquivo, caminho_local)
        except Exception as e:
            raise IOError(f"Erro ao baixar {nome_arquivo} do GitHub. ",
                          "Verifique a conexão ou a URL. Detalhes: {e}")
            
    return caminho_local

def ler_pgm_p2(caminho):
    with open(caminho) as f:
        linhas = [l for l in f.read().split() if l]
    assert linhas[0] == "P2"
    C, L = int(linhas[1]), int(linhas[2])
    maxv = int(linhas[3])
    valores = list(map(int, linhas[4:4 + L * C]))
    return np.array(valores, dtype=np.uint8).reshape(L, C)

# Garantiza la descarga y obtiene la ruta correcta
arq_caso1 = garantir_e_baixar_arquivo("Caso1_Mosaico_Simples.pgm")
arq_caso2 = garantir_e_baixar_arquivo("Caso2_Mosaico_Misto.pgm")

# Lee las matrices PGM
caso1 = ler_pgm_p2(arq_caso1)
caso2 = ler_pgm_p2(arq_caso2)

mm.show(
    [caso1, caso2],
    titles=[
        "Caso 1: Mosaico Sencillo\n(2x2 bloques, 1 muestra/clase)",
        "Caso 2: Mosaico Mixto\n(3x3 bloques, clases repetidas)",
    ],
    cols=2,
    figsize=(8, 4),
)

**Figura 7.7:** Mosaicos de referencia (formato PGM ASCII) utilizados en el EP07_07. Caso 1: cuadrícula 2x2 con una muestra de cada clase. Caso 2: cuadrícula 3x3 con clases repetidas y mayor variabilidad.


In [ ]:
from IPython.display import HTML
HTML('''
<div id="sim-ep0707" style="background-color:#fef9ef;border-radius:18px;border:1px solid #ede6d8;overflow:hidden;margin-top:20px;font-family:sans-serif;">  
<style>
  #sim-ep0707 * { box-sizing: border-box; margin: 0; padding: 0; }
  #sim-ep0707 button { font-size: 11px; padding: 6px 12px; border-radius: 8px; border: 1px solid #ede6d8; background: #f3efe6; color: #5e5a4a; cursor: pointer; transition: all 0.15s ease; font-weight: 600; display: inline-flex; align-items: center; justify-content: center; gap: 5px; }
  #sim-ep0707 button:hover { background: #e8e0cf; }
  #sim-ep0707 button.sim-ep0707_active { background: #26241d !important; border-color: #26241d !important; color: #7ee7c6 !important; }
  .sim-ep0707_pill { font-size: 10px; font-weight: 700; padding: 3px 10px; border-radius: 40px; border: 1px solid #ede6d8; background: #26241d; color: #7ee7c6; font-family: monospace; }
  .sim-ep0707_panel { background: #fafaf7; border: 1px solid #e9e3d3; border-radius: 12px; padding: 12px; }
</style>
  
<!-- Cabeçalho -->
<div style="background:#f1ead7;padding:8px 16px;font-size:11.5px;color:#5e5a4a;border-bottom:1px solid #e4dcc8;display:flex;justify-content:space-between;align-items:center;flex-wrap:wrap;gap:6px;">
  <span style="font-weight:600;color:#26241d;">🎮 Simulador EP07_07: Clasificación de Mosaico mediante LBP + k-NN</span>
  <span class="sim-ep0707_pill">⚫ pipeline completo</span>
</div>


  <div style="padding:20px;background:white;overflow:auto">
    <p style="font-size:11px;color:#777;margin-bottom:14px;text-align:center;">
Mosaico 3x3 de bloques 12x12 (L=C=36). LBP (P=8,R=1) calculado píxel a píxel, con exclusión del borde global.      Ajusta k y la métrica y observa la clasificación de cada bloque frente a 8 prototipos (2 por clase).
   
   </p>
     
<div style="background:#fff3cd;border:1px solid #ffe69c;border-radius:8px;padding:8px 12px;margin-bottom:12px;font-size:11px;color:#7a5c00;">
  ⚠️ Texturas sintéticas generadas por código, no los archivos .pgm reales del EP07_07. Usa este simulador para entender el flujo del algoritmo, no como referencia de dificultad entre las clases.
</div>
     
    <div style="background:#fafafa;border:1px solid #ddd;border-radius:12px;padding:16px;margin-bottom:16px;display:flex;gap:24px;flex-wrap:wrap;align-items:center;">
      <div style="flex:1;min-width:180px;">
        <div style="display:flex;justify-content:space-between;margin-bottom:6px;">
          <label style="font-size:12px;font-weight:bold;color:#2980b9;">k (número de vecinos)</label>
          <span id="ep0707_vl" style="font-family:monospace;font-weight:bold;color:#2980b9;">1</span>
        </div>
<input id="ep0707_sl" style="width:100%;accent-color:#2980b9;" max="5" min="1" step="2" type="range" value="1">
      </div>
      <div>
        <label style="font-size:12px;font-weight:bold;color:#2980b9;display:block;margin-bottom:6px;">Métrica</label>
        <select id="ep0707_metric" style="font-size:12px;padding:4px 8px;border-radius:6px;border:1px solid #ccc;">
          <option value="euclidiana">euclidiana</option>
          <option value="manhattan">manhattan</option>
        </select>
      </div>
    </div>

    <div style="display:flex;gap:20px;flex-wrap:wrap;align-items:flex-start;">
      <canvas id="ep0707_canvas" style="border-radius:8px;border:1px solid #ccc;"></canvas>
      <div id="ep0707_grid" style="flex:1;min-width:220px;display:grid;grid-template-columns:repeat(3,1fr);gap:8px;"></div>
    </div>

    <div id="ep0707_debug" style="margin-top:16px;background:#e3f2fd;border-radius:8px;padding:10px;border:1px solid #bbdefb;font-family:monospace;font-size:11px;color:#1565c0;white-space:pre-line;"></div>
  </div>
</div>
<script>
(function(){
  function init(root){
    if(!root || root.dataset.init) return;
    root.dataset.init = "1";

    var S = 12, G = 3, L = G * S, SCALE = 5;   // bloco maior reduz o vazamento de borda; SCALE ajustado p/ manter o canvas ~180px
    var classesOrder = ["granular", "listrada", "manchada", "xadrez"];
    // Grade 3x3 com classes repetidas, análoga ao Caso 2 do enunciado
    var layout = [
      "granular", "listrada", "manchada",
      "xadrez",   "granular", "manchada",
      "listrada", "xadrez",   "granular"
    ];

    // --- Geração determinística de textura por pixel (didática, não os PGMs reais) ---
    function h(a, b, phase){
      var v = Math.sin((a + phase) * 12.9898 + (b + phase * 0.7) * 78.233 + phase * 3.1) * 43758.5453;
      return v - Math.floor(v);
    }
    function texturePixel(cls, r, c, phase){
      phase = phase || 0;
      switch(cls){
        case "granular": return h(r, c, phase) < 0.5 ? 220 : 30;
        case "listrada": return ((c + Math.floor(phase * 2)) % 4) < 2 ? 220 : 30;
        case "manchada": return h(Math.floor(r / 3), Math.floor(c / 3), phase) < 0.5 ? 200 : 60;
        case "xadrez":   return ((Math.floor(r / 2) + Math.floor(c / 2)) % 2 === 0) ? 230 : 20;
      }
    }

    function buildImage(){
      var img = [];
      for(var r = 0; r < L; r++){
        var row = [];
        for(var c = 0; c < L; c++){
          var bi = Math.floor(r / S), bj = Math.floor(c / S);
          row.push(texturePixel(layout[bi * G + bj], r, c, 0));
        }
        img.push(row);
      }
      return img;
    }

    // --- LBP: P=8, R=1, sentido horário, s_p = 1 se vizinho >= centro ---
    function lbpBin(patch, r, c){
      var center = patch[r][c];
      var neigh = [
        patch[r-1][c-1], patch[r-1][c], patch[r-1][c+1],
        patch[r][c+1],
        patch[r+1][c+1], patch[r+1][c], patch[r+1][c-1],
        patch[r][c-1]
      ];
      var bits = neigh.map(function(v){ return v >= center ? 1 : 0; });
      var trans = 0;
      for(var i = 0; i < 8; i++){ if(bits[i] !== bits[(i+1) % 8]) trans++; }
      if(trans <= 2) return bits.reduce(function(a,b){ return a+b; }, 0); // popcount 0..8
      return 9; // não uniforme
    }

    // Histograma de um patch isolado (usado para gerar protótipos), excluindo apenas a borda do patch
    function computeLBPHist(patch){
      var n = patch.length, m = patch[0].length;
      var hist = new Array(10).fill(0), count = 0;
      for(var r = 1; r < n - 1; r++){
        for(var c = 1; c < m - 1; c++){
          hist[lbpBin(patch, r, c)]++;
          count++;
        }
      }
      for(var k = 0; k < 10; k++) hist[k] = count > 0 ? hist[k] / count : 0;
      return hist;
    }

    // Histogramas por bloco da imagem completa, excluindo só a borda global (item 4/5 do enunciado)
    function computeMosaicHistograms(img){
      var hists = [], counts = [];
      for(var i = 0; i < G*G; i++){ hists.push(new Array(10).fill(0)); counts.push(0); }
      for(var r = 1; r < L - 1; r++){
        for(var c = 1; c < L - 1; c++){
          var bin = lbpBin(img, r, c);
          var idx = Math.floor(r/S) * G + Math.floor(c/S);
          hists[idx][bin]++;
          counts[idx]++;
        }
      }
      for(var b = 0; b < hists.length; b++){
        for(var k = 0; k < 10; k++) hists[b][k] = counts[b] > 0 ? hists[b][k] / counts[b] : 0;
      }
      return hists;
    }

    // --- Protótipos: 2 por classe (N=8), ordem de leitura fixa (usada no desempate) ---
    var prototypes = [];
    classesOrder.forEach(function(cls){
      [0, 5].forEach(function(phase){
        var Sp = S + 2, patch = [];
        for(var r = 0; r < Sp; r++){
          var row = [];
          for(var c = 0; c < Sp; c++) row.push(texturePixel(cls, r, c, phase));
          patch.push(row);
        }
        prototypes.push({ classe: cls, hist: computeLBPHist(patch) });
      });
    });

    function dist(u, v, metric){
      var s = 0;
      for(var i = 0; i < u.length; i++){
        s += metric === "euclidiana" ? (u[i]-v[i])*(u[i]-v[i]) : Math.abs(u[i]-v[i]);
      }
      return metric === "euclidiana" ? Math.sqrt(s) : s;
    }

    // Desempate de distância: ordem de leitura dos protótipos. Desempate de votação: ordem das classes.
    function classify(hist, k, metric){
      var cand = prototypes.map(function(p, idx){ return { classe: p.classe, d: dist(hist, p.hist, metric), idx: idx }; });
      cand.sort(function(a, b){ return a.d !== b.d ? a.d - b.d : a.idx - b.idx; });
      var viz = cand.slice(0, k);
      var votos = {};
      viz.forEach(function(v){ votos[v.classe] = (votos[v.classe] || 0) + 1; });
      var melhor = null, melhorN = -1;
      classesOrder.forEach(function(c){
        var n = votos[c] || 0;
        if(n > melhorN){ melhorN = n; melhor = c; }
      });
      return melhor;
    }

    var canvas = root.querySelector('#ep0707_canvas');
    canvas.width = L * SCALE; canvas.height = L * SCALE;
    var ctx = canvas.getContext('2d');
    var slK = root.querySelector('#ep0707_sl');
    var vlK = root.querySelector('#ep0707_vl');
    var selMetric = root.querySelector('#ep0707_metric');
    var gridEl = root.querySelector('#ep0707_grid');
    var dbg = root.querySelector('#ep0707_debug');

    var img = buildImage();
    var hists = computeMosaicHistograms(img);

    function render(){
      var k = parseInt(slK.value);
      var metric = selMetric.value;
      vlK.textContent = k;

      var preds = [];
      for(var idx = 0; idx < G*G; idx++) preds.push(classify(hists[idx], k, metric));

      var confusion = classesOrder.map(function(){ return new Array(classesOrder.length).fill(0); });
      var acertos = 0;
      for(var i2 = 0; i2 < G*G; i2++){
        var ri = classesOrder.indexOf(layout[i2]);
        var pi = classesOrder.indexOf(preds[i2]);
        confusion[ri][pi]++;
        if(layout[i2] === preds[i2]) acertos++;
      }
      var acc = acertos / (G*G);

      // Desenha a imagem real em tons de cinza
      for(var r = 0; r < L; r++){
        for(var c = 0; c < L; c++){
          var v = img[r][c];
          ctx.fillStyle = 'rgb(' + v + ',' + v + ',' + v + ')';
          ctx.fillRect(c*SCALE, r*SCALE, SCALE, SCALE);
        }
      }
      // Contorna cada bloco: verde = acerto, vermelho = erro
      for(var idx3 = 0; idx3 < G*G; idx3++){
        var bi = Math.floor(idx3 / G), bj = idx3 % G;
        ctx.strokeStyle = (preds[idx3] === layout[idx3]) ? '#10b981' : '#f43f5e';
        ctx.lineWidth = 2;
        ctx.strokeRect(bj*S*SCALE + 1, bi*S*SCALE + 1, S*SCALE - 2, S*SCALE - 2);
      }

      // Grade textual de apoio
      gridEl.innerHTML = '';
      for(var idx4 = 0; idx4 < G*G; idx4++){
        var ok = preds[idx4] === layout[idx4];
        var card = document.createElement('div');
        card.style.cssText = 'border-radius:8px;padding:6px;text-align:center;font-size:10px;border:2px solid ' + (ok ? '#10b981' : '#f43f5e') + ';';
        card.innerHTML = 'Real: ' + layout[idx4] + '<br><b style="color:' + (ok ? '#059669' : '#e11d48') + '">Pred: ' + preds[idx4] + (ok ? ' ✅' : ' ❌') + '</b>';
        gridEl.appendChild(card);
      }

      // Saída no mesmo formato do programa (itens 7-9 do enunciado)
      var linhas = [];
      linhas.push('Classes preditas (ordem de leitura da grade):');
      linhas.push(preds.join(' '));
      linhas.push('');
      linhas.push('Matriz de confusão (linhas=real, colunas=predita; ordem ' + classesOrder.join(',') + '):');
      confusion.forEach(function(lin){ linhas.push(lin.join(' ')); });
      linhas.push('');
      linhas.push('Acuracia: ' + acc.toFixed(4));
      dbg.textContent = linhas.join('\\n');
    }

    slK.addEventListener('input', render);
    selMetric.addEventListener('change', render);
    render();
  }

  function tryInit(){
    var root = document.getElementById('sim-ep0707');
    if(root) init(root); else setTimeout(tryInit, 200);
  }
  tryInit();
})();
</script>
''')

**Figura 7.8:** Simulador EP07_07: Clasificación de un Mosaico de Texturas vía LBP + k-NN


In [ ]:
%%writefile EP07_07.py
# Código Python

In [ ]:
TestSuite("EP07_07.py").run()